# Diabetes MLOps — Guided Master Lab (step-by-step edition)
**Google Colab → Google Drive → MLflow → BentoML → Docker → GitHub Actions → GHCR → Ubuntu**

This is the expanded learning edition of the previous notebook. Keep it open alongside Colab. Each small code cell has a purpose and expected result. Run **one cell at a time** with Shift+Enter. Read the output before moving on; do not use Run all on your first attempt. All five model families and all deployment files remain included.

**Where commands run:** Python cells run in Colab. Markdown terminal blocks marked **Ubuntu** run in a trusted Ubuntu shell, and **GitHub** steps run in a web browser. No credentials in notebook cells. If a cell fails, stop and inspect the error rather than skipping assertions. A fresh Colab session creates a fresh run ID. Mount Drive again after a reset. The notebook does not claim any external deployment has already passed.

**Scope:** educational model; no diagnosis, no real patient input. CI, image publishing and deployment only. No monitoring, Prometheus, Grafana or Kubernetes yet.

**Learning objective:** understand each intermediate data table, each logged model, every Docker layer and every workflow job; then reproduce the deployment on a clean VM without manual rebuilding.

## Contents

- 01 · Install modules
- 02 · Drive + run configuration
- 03 · Data collection and immutable raw storage
- 04 · Data engineering — schema, cleaning and integrity
- 05 · Feature engineering and structural EDA
- 06 · Model initialization, data split and split analysis
- 07 · Development EDA and statistical analysis
- 08 · Baseline model development and metrics
- 09 · Advanced development analysis (out-of-fold)
- 10 · MLflow initialization and five baseline experiment runs
- 11 · Load saved models, independently recompute metrics and compare with MLflow
- 12 · Adaptive retraining — analysis-driven, deterministic policy
- 13 · Consistency, model comparison and freeze final candidate
- 14 · Final evaluation — one frozen candidate, one holdout
- 15 · MLflow Registry and alias
- 16 · BentoML init and registry model import
- 17 · Generate a typed BentoML service in Colab
- 18 · Create deterministic API fixtures and automated tests
- 19 · Start the BentoML server in Colab, send requests and analyze responses
- 20 · Preserve the model and generate serving requirements
- 21 · Docker — layers, image, container and Compose
- 22 · Optional actual Docker build/run in Colab
- 23 · Continuous integration and continuous delivery
- 24 · Deployment mode selection: ephemeral THM VM vs persistent host
- 25 · Final package, integrity checks and backup
- 26 · GitHub upload and CI/CD execution (external steps)
- 27 · Fresh VM deployment (no manual Docker build)
- 28 · Completion checklist and release decision
- References and boundaries

## 01 · Install modules
These are deliberately selected, compatible-generation lab versions, not a claim that they are the latest. The serving requirements are generated from the actually imported environment. If Colab reports a previously loaded package changed, restart the session and run from the top. The installer is idempotent for the pinned versions. Use Python 3.10–3.13; do not silently change sklearn between training and serving.

### Step 001 — Check Python and select packages

**What and why:** sys.executable points to the actual notebook interpreter. Pin MLflow, BentoML and sklearn; the rest have compatible ranges.

**Checkpoint:** Python is 3.10–3.13; an assertion means stop and use a supported runtime.

In [ ]:
import sys, subprocess
from importlib import metadata
assert (3, 10) <= sys.version_info[:2] <= (3, 13), sys.version
packages = ["mlflow==3.5.1", "bentoml==1.4.27", "scikit-learn==1.6.1",
            "pandas>=2.2,<3", "numpy>=1.26,<3", "scipy>=1.12,<2",
            "matplotlib>=3.8", "seaborn>=0.13", "pydantic>=2.7,<3", "requests>=2.31"]

### Step 002 — Install modules

**What and why:** Run pip through the same interpreter, rather than an unrelated terminal Python.

**Checkpoint:** Installation complete; if Colab reports already imported modules changed, restart once.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("Installation complete. Restart the runtime if Colab reports loaded packages changed.")
# Inspect, rather than assuming all unrelated preinstalled Colab packages agree.

### Step 003 — Inspect dependency health

**What and why:** pip check detects inconsistent installed requirements, including unrelated Colab packages.

**Checkpoint:** Review conflicts; do not proceed with a conflict affecting the model or serving libraries.

In [ ]:
check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
print(check.stdout[:3000], check.stderr[:1000])

## 02 · Drive + run configuration
Raw is content-addressed and never rewritten. Processed copies, charts, model artifacts and audit reports are versioned by a run ID. MLflow SQLite lives on local disk (not mounted Drive); use the backup function to create a consistent snapshot on Drive. Artifacts are written to a persistent directory. This is a single-user lab, not a multi-user tracking server.

### Step 004 — Import filesystem tools and mount Drive

**What and why:** Drive is persistent; /content is temporary. Path objects avoid hard-coded shell paths.

**Checkpoint:** Choose your Google account and grant access only to your own Drive.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import os, json, hashlib, shutil, sqlite3, uuid, platform, time, zipfile, io, re
from collections import Counter
from IPython.display import display

drive.mount('/content/drive')

### Step 005 — Create raw, run, and deployment directories

**What and why:** Each execution gets a separate ID. RAW is shared and content-addressed; all processed artifacts are per-run.

**Checkpoint:** The directories exist; run this once per experiment.

In [ ]:
ROOT = Path('/content/drive/MyDrive/Diabetes_MLOps_Master')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:6]
RUN = ROOT / 'runs' / RUN_ID
RAW = ROOT / 'raw'
EXTRACTED = RUN / 'extracted'
PROCESSED = RUN / 'processed'
REPORTS = RUN / 'reports'
ARTIFACTS = RUN / 'mlflow_artifacts'
LOCAL = Path('/content/diabetes_mlops') / RUN_ID
DEPLOY = LOCAL / 'deployment'
for p in [RAW, EXTRACTED, PROCESSED, REPORTS, ARTIFACTS, LOCAL, DEPLOY]:
    p.mkdir(parents=True, exist_ok=True)

### Step 006 — Set reproducibility and quality requirements

**What and why:** Set random seed and educational validation gates before looking at test performance. The Bento store is isolated.

**Checkpoint:** Do not relax the thresholds after viewing holdout results.

In [ ]:
SEED = 42
N_JOBS = 2
MIN_F1, MIN_RECALL, MIN_BALANCED_ACCURACY = .75, .75, .70  # educational gates, set BEFORE holdout
os.environ['BENTOML_HOME'] = str(LOCAL / 'bentoml')
os.environ['BENTOML_DO_NOT_TRACK'] = 'True'
os.environ['GIT_TERMINAL_PROMPT'] = '0'

### Step 007 — Define reusable storage helpers

**What and why:** SHA-256 detects changes; write_json preserves structured metadata; immutable_write refuses to overwrite different bytes.

**Checkpoint:** Printed run paths identify exactly where your work will be saved.

In [ ]:
def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, default=str), encoding='utf-8')

def immutable_write(path, data):
    path = Path(path)
    if path.exists():
        assert path.read_bytes() == data, f'Existing raw file differs: {path}'
    else:
        with path.open('xb') as f:
            f.write(data)

print('Run ID:', RUN_ID, '\nDrive:', RUN, '\nLocal deployment:', DEPLOY)

### Step 008 — Import analysis and machine learning modules

**What and why:** Pandas handles tables, NumPy arrays, matplotlib/seaborn plots, and sklearn provides pipelines and models.

**Checkpoint:** Imports work and sklearn reports 1.6.1.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import sklearn, requests
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder, FunctionTransformer, KBinsDiscretizer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold, cross_validate, cross_val_predict, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix,
    classification_report, make_scorer, roc_curve, precision_recall_curve)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from importlib import metadata
assert sklearn.__version__ == '1.6.1', 'Restart runtime before continuing.'
np.random.seed(SEED)
sns.set_theme(style='whitegrid')

### Step 009 — Capture actual training and serving versions

**What and why:** An environment report is important when moving serialized estimators from Colab to Docker.

**Checkpoint:** Inspect VERSIONS and environment.json; pip freeze is private.

In [ ]:
CORE = ['mlflow','bentoml','scikit-learn','numpy','pandas','scipy','pydantic','joblib','threadpoolctl','cloudpickle']
VERSIONS = {p: metadata.version(p) for p in CORE}
MODEL_PINS = [f'{p}=={VERSIONS[p]}' for p in CORE if p not in {'bentoml','pydantic'}]
SERVING_PINS = [f'{p}=={v}' for p,v in VERSIONS.items()]
write_json(REPORTS / 'environment.json', {'python':sys.version,'platform':platform.platform(),'packages':VERSIONS})
( REPORTS / 'training-pip-freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print(VERSIONS)

### Step 010 — Create a chart-saving helper

**What and why:** All charts are displayed in Colab and saved in the persistent reports directory.

**Checkpoint:** Function definition completes without output.

In [ ]:
def save_plot(name):
    plt.tight_layout()
    plt.savefig(REPORTS / f'{name}.png', dpi=140, bbox_inches='tight')
    plt.show()
    plt.close()

## 03 · Data collection and immutable raw storage
Source: [UCI Early Stage Diabetes Risk Prediction, ID 529](https://archive.ics.uci.edu/dataset/529/early+stage+diabetes+risk+prediction+dataset). The archive is stored exactly as downloaded with a SHA-256 identity. The checksum detects local mutation; it is not an independently published authenticity signature. Read a named CSV member instead of extracting arbitrary archive paths. Do not commit raw records to GitHub.

### Step 011 — Download public UCI dataset

**What and why:** requests retrieves the ZIP over HTTPS and raises an error for unsuccessful responses. A size limit rejects unexpectedly large data.

**Checkpoint:** HTTP request succeeds; if it fails, check connectivity rather than using unrelated data.

In [ ]:
DATA_URL = 'https://archive.ics.uci.edu/static/public/529/early+stage+diabetes+risk+prediction+dataset.zip'
response = requests.get(DATA_URL, timeout=(15, 90))
response.raise_for_status()
blob = response.content
assert len(blob) < 10_000_000, 'Unexpected archive size'

### Step 012 — Preserve original ZIP with a content hash

**What and why:** Identical downloads reuse a raw directory. A changed archive receives a new hash.

**Checkpoint:** Original bytes are stored under raw/<SHA256>/uci_529.zip.

In [ ]:
raw_hash = hashlib.sha256(blob).hexdigest()
raw_dir = RAW / raw_hash
raw_dir.mkdir(exist_ok=True)
raw_zip = raw_dir / 'uci_529.zip'
immutable_write(raw_zip, blob)

### Step 013 — Read the named CSV and preserve the original

**What and why:** Select the expected member rather than extracting arbitrary paths. Save original CSV separately.

**Checkpoint:** One expected CSV member is found.

In [ ]:
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    members = [m for m in z.infolist() if Path(m.filename).name.lower() == 'diabetes_data_upload.csv']
    assert len(members) == 1 and members[0].file_size < 10_000_000
    csv_bytes = z.read(members[0])
raw_csv = raw_dir / 'diabetes_data_upload.csv'
immutable_write(raw_csv, csv_bytes)

### Step 014 — Create an independent working copy and lineage record

**What and why:** The extraction copy is used for analysis. The recorded source, retrieval date and hashes tie later results to the source.

**Checkpoint:** The two CSV hashes match.

In [ ]:
extract_csv = EXTRACTED / raw_csv.name
shutil.copy2(raw_csv, extract_csv)
assert sha256_file(raw_csv) == sha256_file(extract_csv)
manifest = {'source':DATA_URL,'retrieved_utc':datetime.now(timezone.utc).isoformat(),
            'archive_sha256':raw_hash,'csv_sha256':sha256_file(raw_csv), 'raw_path':str(raw_csv)}
write_json(RUN / 'data_manifest.json', manifest)

### Step 015 — Load the copied data into pandas

**What and why:** Load all columns as strings first to inspect source values without premature coercion.

**Checkpoint:** Typically 520 rows and 17 columns; inspect the displayed head.

In [ ]:
raw_df = pd.read_csv(extract_csv, dtype='string')
print('Raw shape:', raw_df.shape, 'SHA256:', manifest['csv_sha256'])
display(raw_df.head())

### Step 016 — Inspect the working data before cleaning

**What and why:** Use info, head, missing counts and exact-duplicate counts to understand the starting shape; the raw copy is read-only by convention.

**Checkpoint:** 17 columns, source categories and any missing/duplicate counts are shown.

In [ ]:
print('Working file:', extract_csv)
print('Shape:', raw_df.shape)
raw_df.info()
display(raw_df.head(5))
display(pd.DataFrame({'missing':raw_df.isna().sum(),'distinct':raw_df.nunique()}))
print('Exact source duplicates:', raw_df.duplicated().sum())

## 04 · Data engineering — schema, cleaning and integrity
The raw file stays untouched. Normalize column names, whitespace, categorical values and target. Invalid rows are quarantined (not guessed). Exact duplicate feature+label rows are removed. Identical features with conflicting labels are reported, kept in the same split group, and never allowed across train and test. This is a feature-profile group, not a verified patient ID; true patient-level leakage cannot be ruled out.

### Step 017 — Declare schema and normalize column names

**What and why:** Define the 16 expected fields, copy the raw DataFrame, and enforce the schema.

**Checkpoint:** All fields present; no missing or unexpected columns.

In [ ]:
FEATURES = ['age','gender','polyuria','polydipsia','sudden_weight_loss','weakness','polyphagia',
 'genital_thrush','visual_blurring','itching','irritability','delayed_healing','partial_paresis',
 'muscle_stiffness','alopecia','obesity']
SYMPTOMS = FEATURES[2:]
TARGET = 'class'
df = raw_df.copy(deep=True)
df.columns = [re.sub(r'[^a-z0-9]+','_',str(c).strip().lower()).strip('_') for c in df.columns]
assert df.columns.is_unique and set(df.columns) == set(FEATURES+[TARGET]), df.columns.tolist()
df = df[FEATURES+[TARGET]].copy()

### Step 018 — Normalize values and data types

**What and why:** Strip whitespace, map blanks and ? to missing, coerce age, and standardize categorical spelling.

**Checkpoint:** Age becomes numeric; categorical values use canonical case.

In [ ]:
for c in df.columns:
    df[c] = df[c].str.strip().replace({'?':pd.NA,'':pd.NA})
df['age'] = pd.to_numeric(df['age'], errors='coerce').astype(float)
for c in FEATURES[1:]+[TARGET]:
    df[c] = df[c].str.lower().str.capitalize()

### Step 019 — Validate values and quarantine bad rows

**What and why:** Check age, gender, label and symptom domains. Invalid records are saved privately; no fabricated labels or clinical values.

**Checkpoint:** Quarantine file exists; inspect the count in the audit below.

In [ ]:
valid = df['age'].between(1,120) & df['gender'].isin(['Male','Female']) & df[TARGET].isin(['Positive','Negative'])
for c in SYMPTOMS:
    valid &= df[c].isin(['Yes','No'])
invalid = df.loc[~valid].copy()
invalid.to_csv(PROCESSED/'quarantine.csv',index=False)
clean = df.loc[valid].copy()

### Step 020 — Deduplicate and encode the target

**What and why:** Identical feature and label records are removed. Negative becomes 0 and Positive becomes 1.

**Checkpoint:** Both classes must remain.

In [ ]:
exact_duplicates = int(clean.duplicated().sum())
clean = clean.drop_duplicates().reset_index(drop=True)
for c in FEATURES[1:]: clean[c] = clean[c].astype(str)
clean[TARGET] = clean[TARGET].map({'Negative':0,'Positive':1}).astype(int)
# Stable profile hash; independent of target, and portable across sessions.

### Step 021 — Identify potentially leaking duplicate profiles

**What and why:** Hash only the feature values so conflicting labels on identical profiles stay together during splits.

**Checkpoint:** The hash is a profile identifier, not proof of patient identity.

In [ ]:
clean['group'] = [hashlib.sha256(json.dumps(list(row),separators=(',',':')).encode()).hexdigest()
                  for row in clean[FEATURES].itertuples(index=False,name=None)]
conflicts = clean.groupby('group')[TARGET].nunique().gt(1)

### Step 022 — Save cleaned dataset and data quality report

**What and why:** Record raw, rejected, duplicate and retained row counts and verify original file integrity.

**Checkpoint:** cleaned.csv and cleaning_audit.json are saved; hashes still match.

In [ ]:
audit = {'raw_rows':len(raw_df),'quarantined':len(invalid),'exact_duplicates_removed':exact_duplicates,
         'clean_rows':len(clean),'conflicting_profiles':int(conflicts.sum()),'unique_profiles':int(clean.group.nunique())}
assert len(clean) > 100 and clean[TARGET].nunique()==2
clean.to_csv(PROCESSED/'cleaned.csv',index=False)
write_json(REPORTS/'cleaning_audit.json',audit)
assert sha256_file(raw_zip)==raw_hash and sha256_file(raw_csv)==manifest['csv_sha256']
display(pd.Series(audit,name='count'))
display(clean[FEATURES].describe(include='all').T)

### Step 023 — Confirm the cleaning accounting

**What and why:** Every removed row must be explained; inspect the cleaned schema and private quarantined records if needed.

**Checkpoint:** Accounting identity is True; cleaned DataFrame contains group plus feature and label columns.

In [ ]:
assert audit['raw_rows'] == audit['quarantined'] + audit['exact_duplicates_removed'] + audit['clean_rows']
print('Row accounting verified')
display(clean.head(3))
print('Clean types:'); print(clean.dtypes)

## 05 · Feature engineering and structural EDA
We preserve the 16-field API contract. Two derived features are calculated inside the sklearn pipeline: symptom burden (Yes count) and age buckets. Scaling and encoding are fitted separately in each training fold; no preprocessing is fitted on holdout. Structural checks can use the complete file; target-conditioned EDA below uses development data only, after the split.

### Step 024 — Profile data structure before fitting

**What and why:** Check dtypes, unique counts and missingness; the structural age histogram is for data validation, not model selection.

**Checkpoint:** No unexpected nulls after quarantine; schema report is saved.

In [ ]:
# Structural analysis only; target labels are not plotted before splitting.
structure = pd.DataFrame({'dtype':clean[FEATURES].dtypes.astype(str),
                          'missing':clean[FEATURES].isna().sum(), 'unique':clean[FEATURES].nunique()})
display(structure)
structure.to_csv(REPORTS/'schema_profile.csv')
plt.figure(figsize=(9,4)); plt.hist(clean['age'],bins=15); plt.title('Structural age distribution'); save_plot('age_structure')

### Step 025 — Define feature engineering inside a pipeline

**What and why:** Scale age, one-hot encode categories, count Yes symptoms, and derive age buckets. Learned parameters are fit inside each training fold.

**Checkpoint:** Function is defined; no transformer has seen the holdout yet.

In [ ]:
def make_preprocessor():
    numeric = Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())])
    categoricals = OneHotEncoder(handle_unknown='ignore',sparse_output=False)
    symptom_count = Pipeline([('encode',OrdinalEncoder(categories=[['No','Yes']]*len(SYMPTOMS),
                           handle_unknown='use_encoded_value',unknown_value=-1)),
                           ('count',FunctionTransformer(np.sum,kw_args={'axis':1,'keepdims':True})),
                           ('scale',StandardScaler())])
    age_bins = KBinsDiscretizer(n_bins=4,encode='onehot-dense',strategy='uniform')
    return ColumnTransformer([('age',numeric,['age']),('categories',categoricals,FEATURES[1:]),
                              ('symptom_burden',symptom_count,SYMPTOMS),('age_buckets',age_bins,['age'])])

## 06 · Model initialization, data split and split analysis
Five families: logistic regression, random forest, gradient boosting, support vector machine and K-nearest neighbors. Reserve one fold as an approximately 20% holdout, then build 5-fold stratified group CV on the remaining development records. Every model gets the same CV splits. The holdout is not used to tune, compare or pick a model. Do not rerun selection after seeing holdout outcomes; a failed gate requires fresh independent evaluation data.

### Step 026 — Initialize five candidate estimators

**What and why:** Start with logistic regression, random forest, gradient boosting, SVM and KNN. clone prevents reusing fitted state.

**Checkpoint:** Five unfitted model templates are available.

In [ ]:
MODELS = {
 'LogisticRegression':LogisticRegression(max_iter=1500,random_state=SEED),
 'RandomForest':RandomForestClassifier(n_estimators=160,random_state=SEED,n_jobs=1),
 'GradientBoosting':GradientBoostingClassifier(random_state=SEED),
 'SVM':SVC(probability=True,random_state=SEED),
 'KNN':KNeighborsClassifier(n_neighbors=7),
}
def make_pipeline(estimator):
    return Pipeline([('features',make_preprocessor()),('model',clone(estimator))])

### Step 027 — Separate features, labels and groups

**What and why:** X holds only serving inputs, y is the target, and groups are used only to keep duplicates together.

**Checkpoint:** No target or group hash appears in model inputs.

In [ ]:
X = clean[FEATURES].copy(); y = clean[TARGET].copy(); groups = clean['group'].copy()

### Step 028 — Create a protected holdout

**What and why:** One fold of a stratified group split is held out; the other four become the development set.

**Checkpoint:** Both subsets contain both classes; zero shared group hashes.

In [ ]:
outer = StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
dev_idx,test_idx = next(outer.split(X,y,groups))
X_dev,y_dev,g_dev = [a.iloc[dev_idx].reset_index(drop=True) for a in (X,y,groups)]
X_test,y_test,g_test = [a.iloc[test_idx].reset_index(drop=True) for a in (X,y,groups)]
assert not set(g_dev)&set(g_test)
assert set(y_dev)=={0,1} and set(y_test)=={0,1}

### Step 029 — Make five repeatable development folds

**What and why:** Every candidate uses the exact same grouped validation folds.

**Checkpoint:** No fold shares a feature group across train and validation.

In [ ]:
cv = list(StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED+1).split(X_dev,y_dev,g_dev))
for tr,va in cv:
    assert not set(g_dev.iloc[tr])&set(g_dev.iloc[va])
    assert set(y_dev.iloc[tr])==set(y_dev.iloc[va])=={0,1}

### Step 030 — Analyze and persist split membership

**What and why:** Compare sample sizes and class proportions, and save split indices for experiment lineage.

**Checkpoint:** Approximately 80/20 split; verify exact counts shown, not assumed.

In [ ]:
split_report = pd.DataFrame([{'split':'development','rows':len(y_dev),'positive_rate':y_dev.mean()},
                            {'split':'holdout','rows':len(y_test),'positive_rate':y_test.mean()}])
display(split_report)
# Persist split membership and folds for reproducibility, only in Drive.
clean.assign(split=np.where(np.isin(np.arange(len(clean)),dev_idx),'development','holdout')).to_csv(PROCESSED/'split_membership.csv',index=False)
write_json(PROCESSED/'split_indices.json',{'dev':dev_idx.tolist(),'holdout':test_idx.tolist(),
           'cv':[{'train':a.tolist(),'validation':b.tolist()} for a,b in cv]})
split_report.to_csv(REPORTS/'split_summary.csv',index=False)

### Step 031 — Inspect feature engineering on one training fold

**What and why:** Fit a disposable transformer on the first CV training fold, then transform a few validation rows. This is a demonstration only; each actual model refits its own transformer.

**Checkpoint:** Feature matrix has more columns than raw inputs, and validation data is transformed without fitting.

In [ ]:
tr_demo, va_demo = cv[0]
pre_demo = make_preprocessor()
pre_demo.fit(X_dev.iloc[tr_demo])
engineered_demo = pre_demo.transform(X_dev.iloc[va_demo[:5]])
print('Input shape:', X_dev.iloc[va_demo[:5]].shape)
print('Output shape:', engineered_demo.shape)
display(pd.DataFrame(engineered_demo).round(3))
display(pd.DataFrame({'transformer':[name for name, _, _ in pre_demo.transformers_],
                      'output_columns':[str(pre_demo.output_indices_[name]) for name, _, _ in pre_demo.transformers_]}))

## 07 · Development EDA and statistical analysis
Look at class balance, missingness, age/gender distribution, symptom prevalence, associations and engineered burden. Associations are descriptive, not causal. Avoid interpreting a small subgroup as a clinical result. Raw symptoms are retained for the service; the engineered features are produced automatically at inference.

### Step 032 — Explore class, age, gender and symptom burden

**What and why:** Use only development records for target-aware EDA. Study the distributions before model evaluation.

**Checkpoint:** Four plots appear; look for class imbalance and overlapping distributions.

In [ ]:
eda = X_dev.copy(); eda['target']=y_dev
eda['symptom_burden']=eda[SYMPTOMS].eq('Yes').sum(axis=1)
eda['age_band']=pd.cut(eda.age,[0,30,45,60,120],include_lowest=True)
fig,axes=plt.subplots(2,2,figsize=(12,8))
sns.countplot(data=eda,x='target',ax=axes[0,0]); axes[0,0].set_title('Development class balance')
axes[0,1].hist([eda.loc[eda.target==c,'age'] for c in [0,1]],bins=15,stacked=True,label=['Negative','Positive']); axes[0,1].legend(); axes[0,1].set_title('Age by class')
sns.countplot(data=eda,x='gender',hue='target',ax=axes[1,0]); axes[1,0].set_title('Gender by class')
sns.boxplot(data=eda,x='target',y='symptom_burden',ax=axes[1,1]); axes[1,1].set_title('Symptom burden')
save_plot('development_eda')

### Step 033 — Measure symptom associations by class

**What and why:** Compute Yes proportion separately for positive and negative records and sort by difference.

**Checkpoint:** Heatmap and CSV show which symptoms are associated with the label; not causal effects.

In [ ]:
prevalence = pd.DataFrame({c:eda.groupby('target')[c].apply(lambda x:x.eq('Yes').mean()) for c in SYMPTOMS}).T
prevalence.columns=['negative','positive']; prevalence['difference']=prevalence.positive-prevalence.negative
prevalence=prevalence.sort_values('difference',ascending=False)
display(prevalence)
prevalence.to_csv(REPORTS/'symptom_prevalence.csv')
plt.figure(figsize=(8,7)); sns.heatmap(prevalence[['negative','positive']],annot=True,fmt='.2f',cmap='Blues'); save_plot('symptom_prevalence')

### Step 034 — Explore feature correlations

**What and why:** Spearman rank association on binary encodings and age helps identify redundant or related variables.

**Checkpoint:** Interpret sign and magnitude without treating correlation as causation.

In [ ]:
encoded = pd.concat([eda[['age','target','symptom_burden']],eda[SYMPTOMS].eq('Yes').astype(int)],axis=1)
plt.figure(figsize=(12,10)); sns.heatmap(encoded.corr(method='spearman'),cmap='coolwarm',center=0); save_plot('spearman_associations')

### Step 035 — Compare development subgroups

**What and why:** Age band and gender summaries show counts and observed positive proportions.

**Checkpoint:** Small subgroups are uncertain; do not infer demographic risk in the population.

In [ ]:
groups_eda=eda.groupby(['age_band','gender'],observed=True).target.agg(['size','mean']).rename(columns={'mean':'positive_rate'})
display(groups_eda); groups_eda.to_csv(REPORTS/'development_subgroups.csv')

### Step 036 — Run a more detailed development-data analysis

**What and why:** Compare distributions by class and counts of Yes answers; compute symptom co-occurrence. This is observational EDA, not a clinical risk calculator.

**Checkpoint:** Tables and charts reveal associations and sparse feature combinations.

In [ ]:
display(eda.groupby('target').age.describe())
display(pd.crosstab(eda.age_band, eda.target, margins=True))
counts = eda[SYMPTOMS].eq('Yes').sum().sort_values()
plt.figure(figsize=(8,6)); counts.plot.barh(); plt.xlabel('Yes count in development set'); save_plot('symptom_counts')
flags = eda[SYMPTOMS].eq('Yes').astype(int)
cooccurrence = flags.T @ flags
plt.figure(figsize=(10,8)); sns.heatmap(cooccurrence, cmap='Blues'); plt.title('Symptom co-occurrence'); save_plot('symptom_cooccurrence')

## 08 · Baseline model development and metrics
Accuracy, balanced accuracy, precision, recall, F1, ROC-AUC and average precision. Primary rank is mean validation F1, then recall, then lower variability. Recall is important here because false negatives are relevant in a screening-style exercise. The default decision rule is each estimator's `predict`; probabilities are diagnostic only, not clinically calibrated risks. Train scores diagnose potential overfit; validation scores drive decisions.

### Step 037 — Define metrics and reusable CV evaluator

**What and why:** Accuracy measures overall correctness; balanced accuracy averages class recalls; F1 balances precision and recall. Save each fold and train-validation gap.

**Checkpoint:** Functions defined; training has not started in this cell.

In [ ]:
SCORING = {'accuracy':'accuracy','balanced_accuracy':'balanced_accuracy',
           'precision':make_scorer(precision_score,zero_division=0),
           'recall':make_scorer(recall_score,zero_division=0),
           'f1':make_scorer(f1_score,zero_division=0),'roc_auc':'roc_auc','average_precision':'average_precision'}

def cv_summary(name, pipe, splits=cv):
    res = cross_validate(pipe,X_dev,y_dev,cv=splits,scoring=SCORING,return_train_score=True,
                         n_jobs=N_JOBS,error_score='raise')
    out={'model':name}
    for key in SCORING:
        out[f'cv_{key}_mean']=float(np.mean(res[f'test_{key}']))
        out[f'cv_{key}_std']=float(np.std(res[f'test_{key}'],ddof=1))
        out[f'train_{key}_mean']=float(np.mean(res[f'train_{key}']))
    out['f1_gap']=out['train_f1_mean']-out['cv_f1_mean']
    out['fit_seconds_mean']=float(np.mean(res['fit_time']))
    folds = pd.DataFrame({k:v for k,v in res.items()}); folds.insert(0,'fold',range(1,len(folds)+1))
    return out,folds

def rank(rows):
    return pd.DataFrame(rows).sort_values(['cv_f1_mean','cv_recall_mean','cv_f1_std','model'],
                                          ascending=[False,False,True,True]).reset_index(drop=True)

### Step 038 — Train and evaluate all five models

**What and why:** For every estimator, fit and validate on five folds, then fit a development-only copy for saving.

**Checkpoint:** Five model names with CV F1 and recall; results are computed, never hardcoded.

In [ ]:
baseline_rows=[]; baseline_folds={}; baseline_models={}

def train_one_baseline(name):
    est = MODELS[name]
    pipe=make_pipeline(est)
    row,folds=cv_summary(name,pipe)
    baseline_rows.append(row); baseline_folds[name]=folds
    baseline_models[name]=pipe.fit(X_dev,y_dev)
    folds.to_csv(REPORTS/f'baseline_{name}_folds.csv',index=False)
    print(name, 'CV F1:',round(row['cv_f1_mean'],4),'Recall:',round(row['cv_recall_mean'],4))

#### Train baseline: LogisticRegression
Run this model independently. Inspect the printed result or logged model reference before the next model. If the cell fails, fix the error rather than proceeding. Avoid rerunning a successful cell because it can duplicate runs or list entries.

In [ ]:
train_one_baseline('LogisticRegression')

#### Train baseline: RandomForest
Run this model independently. Inspect the printed result or logged model reference before the next model. If the cell fails, fix the error rather than proceeding. Avoid rerunning a successful cell because it can duplicate runs or list entries.

In [ ]:
train_one_baseline('RandomForest')

#### Train baseline: GradientBoosting
Run this model independently. Inspect the printed result or logged model reference before the next model. If the cell fails, fix the error rather than proceeding. Avoid rerunning a successful cell because it can duplicate runs or list entries.

In [ ]:
train_one_baseline('GradientBoosting')

#### Train baseline: SVM
Run this model independently. Inspect the printed result or logged model reference before the next model. If the cell fails, fix the error rather than proceeding. Avoid rerunning a successful cell because it can duplicate runs or list entries.

In [ ]:
train_one_baseline('SVM')

#### Train baseline: KNN
Run this model independently. Inspect the printed result or logged model reference before the next model. If the cell fails, fix the error rather than proceeding. Avoid rerunning a successful cell because it can duplicate runs or list entries.

In [ ]:
train_one_baseline('KNN')

### Step 039 — Rank baseline models

**What and why:** Rank by F1, then recall, then lower fold variability. Plot mean F1 with fold standard deviation.

**Checkpoint:** A five-row ranked table; bars are fold variability, not confidence intervals.

In [ ]:
baseline_table=rank(baseline_rows); display(baseline_table.round(4)); baseline_table.to_csv(REPORTS/'baseline_comparison.csv',index=False)
plt.figure(figsize=(10,4)); plt.bar(baseline_table.model,baseline_table.cv_f1_mean,yerr=baseline_table.cv_f1_std,capsize=4)
plt.ylim(0,1.08); plt.xticks(rotation=20); plt.ylabel('CV F1 ± fold SD'); save_plot('baseline_f1')

### Step 040 — Compare all five baseline metrics side by side

**What and why:** Look beyond accuracy: false-negative-sensitive recall, precision, F1, ROC-AUC and train-CV gap describe different behavior.

**Checkpoint:** Five rows, no hardcoded winning algorithm.

In [ ]:
cols=['model','cv_accuracy_mean','cv_balanced_accuracy_mean','cv_precision_mean','cv_recall_mean','cv_f1_mean','cv_roc_auc_mean','cv_f1_std','f1_gap']
display(baseline_table[cols].round(4))
ax=baseline_table.set_index('model')[['cv_precision_mean','cv_recall_mean','cv_f1_mean']].plot.bar(figsize=(11,4),ylim=(0,1.05))
ax.set_ylabel('Mean CV score'); save_plot('baseline_metric_comparison')

## 09 · Advanced development analysis (out-of-fold)
Each row is predicted by a model not fitted on that row or its duplicate profile. Analyze errors, probability discrimination and calibration. SVM probability estimates and the hard predictions may disagree; evaluate both intentionally. These are development diagnostics, not an unbiased final estimate after selection.

### Step 041 — Define advanced classification metrics

**What and why:** Add ROC-AUC and average precision for ranking, and Brier loss for probabilistic accuracy.

**Checkpoint:** Metric function returns a dictionary.

In [ ]:
def metrics(y_true,pred,prob):
    return {'accuracy':accuracy_score(y_true,pred),'balanced_accuracy':balanced_accuracy_score(y_true,pred),
            'precision':precision_score(y_true,pred,zero_division=0),'recall':recall_score(y_true,pred,zero_division=0),
            'f1':f1_score(y_true,pred,zero_division=0),'roc_auc':roc_auc_score(y_true,prob),
            'average_precision':average_precision_score(y_true,prob),'brier':brier_score_loss(y_true,prob)}

### Step 042 — Define out-of-fold error and calibration analysis

**What and why:** Obtain predictions on unseen validation folds, classify FP/FN, draw ROC/PR/calibration charts, and inspect subgroups.

**Checkpoint:** Function defined; no holdout values used.

In [ ]:
def analyze_oof(name,pipe,prefix):
    p=cross_val_predict(pipe,X_dev,y_dev,cv=cv,method='predict',n_jobs=N_JOBS)
    prob=cross_val_predict(pipe,X_dev,y_dev,cv=cv,method='predict_proba',n_jobs=N_JOBS)[:,1]
    vals=metrics(y_dev,p,prob)
    errors=X_dev.copy(); errors['actual']=y_dev; errors['predicted']=p; errors['score']=prob
    errors['error_type']=np.select([(y_dev==1)&(p==0),(y_dev==0)&(p==1)],['FN','FP'],default='correct')
    errors.to_csv(PROCESSED/f'{prefix}_oof_predictions.csv',index=False)
    print(name,vals); print(classification_report(y_dev,p,zero_division=0))
    fig,ax=plt.subplots(2,2,figsize=(10,8))
    sns.heatmap(confusion_matrix(y_dev,p,labels=[0,1]),annot=True,fmt='d',ax=ax[0,0]); ax[0,0].set(title='OOF confusion matrix',xlabel='Predicted',ylabel='Actual')
    fpr,tpr,_=roc_curve(y_dev,prob); ax[0,1].plot(fpr,tpr); ax[0,1].plot([0,1],[0,1],'--'); ax[0,1].set(title='ROC',xlabel='False positive rate',ylabel='True positive rate')
    prec,rec,_=precision_recall_curve(y_dev,prob); ax[1,0].plot(rec,prec); ax[1,0].axhline(y_dev.mean(),linestyle='--'); ax[1,0].set(title='Precision-recall',xlabel='Recall',ylabel='Precision')
    true,mean=calibration_curve(y_dev,prob,n_bins=5,strategy='quantile'); ax[1,1].plot(mean,true,'o-'); ax[1,1].plot([0,1],[0,1],'--'); ax[1,1].set(title='Calibration (diagnostic)',xlabel='Mean score',ylabel='Observed proportion')
    save_plot(prefix+'_diagnostics')
    subgroup=[]
    for field, labels in [('gender',X_dev.gender),('age_band',pd.cut(X_dev.age,[0,30,45,60,120],include_lowest=True))]:
        for label in labels.unique():
            mask=(labels==label).to_numpy(); yt=y_dev.to_numpy()[mask]; yp=p[mask]
            subgroup.append({'field':field,'group':str(label),'n':len(yt),'positives':int(yt.sum()),'false_negatives':int(((yt==1)&(yp==0)).sum()),
                             'recall':recall_score(yt,yp,zero_division=0) if yt.sum() else np.nan,
                             'f1':f1_score(yt,yp,zero_division=0) if yt.sum() else np.nan})
    subgroup=pd.DataFrame(subgroup); display(subgroup); subgroup.to_csv(REPORTS/f'{prefix}_subgroups.csv',index=False)
    display(errors.error_type.value_counts()); return vals,errors

### Step 043 — Investigate the top baseline

**What and why:** Review false negatives, false positives and estimated calibration before selecting a tuning strategy.

**Checkpoint:** Confusion matrix, classification report, curves and subgroup tables appear.

In [ ]:
baseline_winner=baseline_table.iloc[0].model
base_oof,base_errors=analyze_oof(baseline_winner,baseline_models[baseline_winner],'baseline_winner')

## 10 · MLflow initialization and five baseline experiment runs
An MLflow **experiment** groups **runs**. Each model has one baseline run storing fitted model, hyperparameters, fold metrics, aggregate metrics and data lineage. We additionally log metrics to a separate metrics-audit experiment (five runs). Use MLflow 3 returned `model_uri`, not an assumed run artifact path. We explicitly declare model dependencies to avoid recording unrelated Colab packages. Only local trusted own model serialization is used.

### Step 044 — Initialize MLflow tracking and registry

**What and why:** Use local SQLite for metadata, Drive for artifacts, and one client to query runs and registry.

**Checkpoint:** MLflow version and tracking URI are valid.

In [ ]:
import mlflow, mlflow.sklearn
from mlflow import MlflowClient
from mlflow.models import infer_signature
assert mlflow.__version__=='3.5.1'
DB=LOCAL/'mlflow.db'
TRACKING_URI=f'sqlite:///{DB}'
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_registry_uri(TRACKING_URI)
os.environ['MLFLOW_TRACKING_URI']=TRACKING_URI
client=MlflowClient()

### Step 045 — Create separate experiment groups

**What and why:** An experiment is a collection of runs, not a single model. Create baseline, audit, tuning and final-validation groups.

**Checkpoint:** Four experiment IDs are created.

In [ ]:
def experiment(label):
    name=f'Diabetes/{RUN_ID}/{label}'
    existing=client.get_experiment_by_name(name)
    return existing.experiment_id if existing else client.create_experiment(name,artifact_location=(ARTIFACTS/label).as_uri())
BASE_EXP=experiment('01-baseline-models')
AUDIT_EXP=experiment('02-baseline-metrics-audit')
TUNE_EXP=experiment('03-adaptive-retraining')
FINAL_EXP=experiment('04-final-validation')

### Step 046 — Define SQLite backup

**What and why:** Use SQLite backup instead of copying an active DB; Drive snapshot and artifact paths support recovery.

**Checkpoint:** Backup helper defined.

In [ ]:
def checkpoint_mlflow():
    target=RUN/'mlflow.sqlite'
    tmp=LOCAL/'mlflow-backup.sqlite'
    with sqlite3.connect(DB) as src, sqlite3.connect(tmp) as dest:
        src.backup(dest)
    shutil.copy2(tmp,target)
    write_json(RUN/'mlflow_restore.json',{'original_tracking_uri':TRACKING_URI,'backup':str(target),'artifact_root':str(ARTIFACTS),
               'python':sys.version,'restore_note':'Mount Drive at same path; copy SQLite backup to local disk; set tracking and registry URIs to copied DB.'})
    print('MLflow database checkpoint:',target)

# Synthetic contract example, no raw observation stored as model input example.

### Step 047 — Create a non-patient schema example

**What and why:** The synthetic profile demonstrates the raw serving contract without publishing actual dataset rows.

**Checkpoint:** One 16-column DataFrame.

In [ ]:
fixture_record={'age':45.0,'gender':'Female',**{s:'No' for s in SYMPTOMS}}
fixture_record.update(polyuria='Yes',polydipsia='Yes',weakness='Yes')
fixture_frame=pd.DataFrame([fixture_record],columns=FEATURES)
fixture_frame['age']=fixture_frame.age.astype('float64')

### Step 048 — Define MLflow model and metrics logging

**What and why:** Log parameters, aggregate metrics, fold values, data lineage and model signature as one model run.

**Checkpoint:** Helper defined; inspect the distinction between run ID and model URI.

In [ ]:
def log_model_run(exp,stage,name,pipe,row,folds,extra=None):
    with mlflow.start_run(experiment_id=exp,run_name=f'{stage}-{name}') as run:
        mlflow.set_tags({'stage':stage,'model_family':name,'dataset_sha256':manifest['csv_sha256'],
                         'split_id':sha256_file(PROCESSED/'split_indices.json'),'purpose':'educational'})
        mlflow.log_params({k:str(v) for k,v in pipe.get_params().items() if k.startswith('model__')})
        mlflow.log_metrics({k:float(v) for k,v in row.items() if k!='model'})
        for _,fold in folds.iterrows():
            mlflow.log_metrics({k:float(v) for k,v in fold.items() if k.startswith(('test_','train_'))},step=int(fold['fold']))
        mlflow.log_dict(manifest,'lineage/data_manifest.json')
        mlflow.log_dict({'folds':len(cv),'seed':SEED,'grouping':'raw feature-profile SHA256'},'lineage/evaluation.json')
        if extra: mlflow.log_dict(extra,'analysis/retraining_policy.json')
        info=mlflow.sklearn.log_model(pipe,name='classifier',signature=infer_signature(fixture_frame,pipe.predict(fixture_frame)),
                                     input_example=fixture_frame,pip_requirements=MODEL_PINS)
        return {'run_id':run.info.run_id,'model_uri':info.model_uri}

### Step 049 — Record five baseline model runs

**What and why:** Each trained pipeline is serialized to MLflow with its own run and URI.

**Checkpoint:** Five model references and a SQLite checkpoint.

In [ ]:
baseline_refs={}

def log_one_baseline(name):
    row = next(r for r in baseline_rows if r["model"] == name)
    name=row['model']; baseline_refs[name]=log_model_run(BASE_EXP,'baseline',name,baseline_models[name],row,baseline_folds[name])

#### Log baseline: LogisticRegression
Run this model independently. Inspect the printed result or logged model reference before the next model. If the cell fails, fix the error rather than proceeding. Avoid rerunning a successful cell because it can duplicate runs or list entries.

In [ ]:
log_one_baseline('LogisticRegression')
print(baseline_refs['LogisticRegression'])

#### Log baseline: RandomForest
Run this model independently. Inspect the printed result or logged model reference before the next model. If the cell fails, fix the error rather than proceeding. Avoid rerunning a successful cell because it can duplicate runs or list entries.

In [ ]:
log_one_baseline('RandomForest')
print(baseline_refs['RandomForest'])

#### Log baseline: GradientBoosting
Run this model independently. Inspect the printed result or logged model reference before the next model. If the cell fails, fix the error rather than proceeding. Avoid rerunning a successful cell because it can duplicate runs or list entries.

In [ ]:
log_one_baseline('GradientBoosting')
print(baseline_refs['GradientBoosting'])

#### Log baseline: SVM
Run this model independently. Inspect the printed result or logged model reference before the next model. If the cell fails, fix the error rather than proceeding. Avoid rerunning a successful cell because it can duplicate runs or list entries.

In [ ]:
log_one_baseline('SVM')
print(baseline_refs['SVM'])

#### Log baseline: KNN
Run this model independently. Inspect the printed result or logged model reference before the next model. If the cell fails, fix the error rather than proceeding. Avoid rerunning a successful cell because it can duplicate runs or list entries.

In [ ]:
log_one_baseline('KNN')
print(baseline_refs['KNN'])

#### Verify completion of all five model runs

In [ ]:
print(baseline_refs)
checkpoint_mlflow()

### Step 050 — Inspect MLflow runs directly

**What and why:** Query tracking metadata, not your in-memory metric table. A run ID identifies an execution; the model URI identifies a loadable model.

**Checkpoint:** Five baseline run entries.

In [ ]:
runs_view = mlflow.search_runs(experiment_ids=[BASE_EXP])
display(runs_view[['run_id','status','tags.model_family','metrics.cv_f1_mean']])
assert len(baseline_refs) == 5

## 11 · Load saved models, independently recompute metrics and compare with MLflow
Load each logged model using the returned URI. Verify hard predictions and probabilities against the in-memory fitted model. Recompute CV from cloned loaded estimators and compare stored means with a strict tolerance; this is reproducibility evidence, not additional independent validation. Metrics audit runs link to the source model run.

### Step 051 — Define saved-model consistency audit

**What and why:** Load serialized models, compare labels and probabilities on development inputs, and recompute cross-validation from a fresh clone.

**Checkpoint:** Helper enforces deterministic metrics within tolerance.

In [ ]:
def reload_and_audit(refs, fitted, rows, exp=None):
    audits=[]; loaded={}; lookup={r['model']:r for r in rows}
    for name,ref in refs.items():
        p=mlflow.sklearn.load_model(ref['model_uri']); loaded[name]=p
        hard=bool(np.array_equal(p.predict(X_dev),fitted[name].predict(X_dev)))
        prob=bool(np.allclose(p.predict_proba(X_dev),fitted[name].predict_proba(X_dev),rtol=1e-8,atol=1e-10))
        fresh,_=cv_summary(name,clone(p))
        saved=client.get_run(ref['run_id']).data.metrics
        keys=[f'cv_{key}_mean' for key in SCORING]
        diff=max(abs(fresh[k]-saved[k]) for k in keys)
        row={'model':name,'hard_equal':hard,'probability_equal':prob,'max_cv_metric_delta':diff,
             'recorded_f1':saved['cv_f1_mean'],'recomputed_f1':fresh['cv_f1_mean']}
        audits.append(row); assert hard and prob and diff<1e-7, row
        if exp is not None:
            with mlflow.start_run(experiment_id=exp,run_name=f'audit-{name}'):
                mlflow.set_tags({'source_run_id':ref['run_id'],'source_model_uri':ref['model_uri']})
                mlflow.log_metrics({k:float(v) for k,v in row.items() if k!='model'})
                mlflow.log_metrics({f'recomputed_{k}':fresh[k] for k in keys})
    return loaded,pd.DataFrame(audits)

### Step 052 — Reload and log five independent metrics audits

**What and why:** Audit runs reference the original model runs. Query MLflow itself to compare saved metrics.

**Checkpoint:** Five audit rows, hard_equal and probability_equal True, metric deltas near zero.

In [ ]:
loaded_baseline,baseline_audit=reload_and_audit(baseline_refs,baseline_models,baseline_rows,AUDIT_EXP)
display(baseline_audit); baseline_audit.to_csv(REPORTS/'baseline_reload_audit.csv',index=False)
recorded=mlflow.search_runs(experiment_ids=[BASE_EXP],order_by=['metrics.cv_f1_mean DESC'])
display(recorded[['run_id','tags.model_family','metrics.cv_f1_mean','metrics.cv_recall_mean','metrics.cv_f1_std']])
checkpoint_mlflow()

## 12 · Adaptive retraining — analysis-driven, deterministic policy
A rules-based automation (not an AI diagnosis): look at F1 train/validation gap, development recall, class ratio and OOF false negatives. If models show overfit, include stronger regularization. If recall or class balance is weak, search cost-sensitive weights where supported. Search bounded hyperparameter grids for all five families. Compare baseline against the tuned candidate on the same development folds; retain baseline parameters if tuning is worse. Retain the entire preprocessing pipeline. The holdout remains untouched.

### Step 053 — Derive automated retraining rules

**What and why:** Use development class balance, missed positives and train-validation F1 gaps to choose search spaces; policy is deterministic, not an LLM.

**Checkpoint:** Five policy rows with parameter grids. No holdout inspection.

In [ ]:
rows_by_name={r['model']:r for r in baseline_rows}
imbalance = min(y_dev.mean(),1-y_dev.mean()) < .40
need_recall = (base_errors.error_type.eq('FN').sum() > 0 and (base_oof['recall']<.90 or imbalance))
policy={}
grids={}
for name,row in rows_by_name.items():
    overfit=row['f1_gap']>.08
    weights=[None,'balanced'] if (need_recall or imbalance or row['cv_recall_mean']<.90) else [None]
    policy[name]={'overfit_gap':row['f1_gap'],'strong_regularization':overfit,'class_weight_options':weights,
                  'development_imbalance':bool(imbalance),'recall_priority':bool(need_recall)}
    if name=='LogisticRegression': grid={'model__C':[.03,.1,1] if overfit else [.3,1,3,10], 'model__class_weight':weights}
    elif name=='RandomForest': grid={'model__n_estimators':[120,200],'model__max_depth':[4,8,12] if overfit else [6,12,None], 'model__min_samples_leaf':[2,4] if overfit else [1,2], 'model__class_weight':weights}
    elif name=='GradientBoosting': grid={'model__n_estimators':[60,120], 'model__max_depth':[1,2] if overfit else [2,3], 'model__learning_rate':[.05,.1]}
    elif name=='SVM': grid={'model__C':[.1,1,3] if overfit else [1,3,10],'model__gamma':['scale',.03],'model__class_weight':weights}
    else: grid={'model__n_neighbors':[9,13,17] if overfit else [3,5,7,11],'model__weights':['uniform','distance'],'model__p':[1,2]}
    grids[name]=grid; policy[name]['parameter_grid']=grid
write_json(REPORTS/'retraining_policy.json',policy)
display(pd.DataFrame(policy).T)

### Step 054 — Tune and retrain five models

**What and why:** GridSearchCV evaluates candidate hyperparameters using the same development folds. Retain baseline parameters if the grid result is worse by the stated rule.

**Checkpoint:** Five trained models and five new MLflow runs; this can be slow.

In [ ]:
tuned_rows=[]; tuned_models={}; tuned_refs={}; tuned_folds={}

def tune_one_model(name):
    est = MODELS[name]
    search=GridSearchCV(make_pipeline(est),grids[name],cv=cv,scoring=SCORING,refit='f1',
                        n_jobs=N_JOBS,error_score='raise',return_train_score=True)
    search.fit(X_dev,y_dev)
    pd.DataFrame(search.cv_results_).to_csv(REPORTS/f'grid_{name}.csv',index=False)
    candidate=clone(search.best_estimator_)
    row,folds=cv_summary(name,candidate)
    baseline=rows_by_name[name]
    # F1 is primary; recall is deterministic tie-breaker.
    if (row['cv_f1_mean'],row['cv_recall_mean']) < (baseline['cv_f1_mean'],baseline['cv_recall_mean']):
        candidate=make_pipeline(est); row,folds=cv_summary(name,candidate); decision='retain_baseline_parameters'
    else: decision='use_grid_parameters'
    pipe=candidate.fit(X_dev,y_dev)
    tuned_models[name]=pipe; tuned_rows.append(row); tuned_folds[name]=folds
    folds.to_csv(REPORTS/f'retrained_{name}_folds.csv',index=False)
    rationale={**policy[name],'decision':decision,'grid_best_params':search.best_params_,
               'baseline_f1':baseline['cv_f1_mean'],'chosen_f1':row['cv_f1_mean']}
    tuned_refs[name]=log_model_run(TUNE_EXP,'retrained',name,pipe,row,folds,rationale)
    print(name,decision,round(row['cv_f1_mean'],4))

#### Tune and log: LogisticRegression
Run this model independently. Inspect the printed result or logged model reference before the next model. If the cell fails, fix the error rather than proceeding. Avoid rerunning a successful cell because it can duplicate runs or list entries.

In [ ]:
tune_one_model('LogisticRegression')

#### Tune and log: RandomForest
Run this model independently. Inspect the printed result or logged model reference before the next model. If the cell fails, fix the error rather than proceeding. Avoid rerunning a successful cell because it can duplicate runs or list entries.

In [ ]:
tune_one_model('RandomForest')

#### Tune and log: GradientBoosting
Run this model independently. Inspect the printed result or logged model reference before the next model. If the cell fails, fix the error rather than proceeding. Avoid rerunning a successful cell because it can duplicate runs or list entries.

In [ ]:
tune_one_model('GradientBoosting')

#### Tune and log: SVM
Run this model independently. Inspect the printed result or logged model reference before the next model. If the cell fails, fix the error rather than proceeding. Avoid rerunning a successful cell because it can duplicate runs or list entries.

In [ ]:
tune_one_model('SVM')

#### Tune and log: KNN
Run this model independently. Inspect the printed result or logged model reference before the next model. If the cell fails, fix the error rather than proceeding. Avoid rerunning a successful cell because it can duplicate runs or list entries.

In [ ]:
tune_one_model('KNN')

### Step 055 — Compare before and after retraining

**What and why:** Join each model family and calculate metric differences. Selection CV scores are exploratory after tuning and may be optimistic.

**Checkpoint:** Five-row before/after table and comparison plot.

In [ ]:
tuned_table=rank(tuned_rows)
comparison=baseline_table.merge(tuned_table,on='model',suffixes=('_baseline','_retrained'))
for k in ['cv_f1_mean','cv_recall_mean','cv_roc_auc_mean','cv_balanced_accuracy_mean']:
    comparison[k+'_delta']=comparison[k+'_retrained']-comparison[k+'_baseline']
comparison.to_csv(REPORTS/'before_after_retraining.csv',index=False)
display(comparison[['model','cv_f1_mean_baseline','cv_f1_mean_retrained','cv_f1_mean_delta','cv_recall_mean_delta','cv_roc_auc_mean_delta']].round(4))
plt.figure(figsize=(10,4)); x=np.arange(len(comparison)); plt.bar(x-.18,comparison.cv_f1_mean_baseline,.36,label='baseline'); plt.bar(x+.18,comparison.cv_f1_mean_retrained,.36,label='retrained'); plt.xticks(x,comparison.model,rotation=20); plt.legend(); plt.ylim(0,1.05); save_plot('baseline_vs_retraining')
checkpoint_mlflow()

## 13 · Consistency, model comparison and freeze final candidate
Reload all five retrained models and compare predictions and recorded metrics. Check repeated group CV on the same development dataset with alternative split seeds. This estimates sensitivity to splits, not independent test performance, because hyperparameters were selected on development data. Compute permutation importance on held-out CV folds (raw features, not training rows). Correlated symptoms may share importance. The final winner is frozen by the predefined ranking, not by the test set.

### Step 056 — Reload and verify retrained models

**What and why:** Repeat serialized prediction and metric consistency checks and query MLflow results.

**Checkpoint:** Five reloaded models agree with their fitted copies.

In [ ]:
loaded_tuned,tuned_audit=reload_and_audit(tuned_refs,tuned_models,tuned_rows)
tuned_audit.to_csv(REPORTS/'retrained_reload_audit.csv',index=False); display(tuned_audit)
recorded_tuned=mlflow.search_runs(experiment_ids=[TUNE_EXP],order_by=['metrics.cv_f1_mean DESC'])
display(recorded_tuned[['run_id','tags.model_family','metrics.cv_f1_mean','metrics.cv_recall_mean','metrics.f1_gap']])
display(tuned_table[['model','cv_f1_mean','cv_recall_mean','cv_f1_std','cv_roc_auc_mean','f1_gap']].round(4))

### Step 057 — Select the development winner

**What and why:** Select only from the ranked development table and keep its saved model reference.

**Checkpoint:** Winner has a model URI and run ID.

In [ ]:
winner=tuned_table.iloc[0].model
winner_model=loaded_tuned[winner]
winner_ref=tuned_refs[winner]

### Step 058 — Check model stability across three seeds

**What and why:** Repeat grouped cross-validation with different fold assignments to assess sensitivity.

**Checkpoint:** Three rows of F1, recall and variability; results are not independent test estimates.

In [ ]:
repeat_rows=[]
for seed in [123,456,789]:
    splits=list(StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=seed).split(X_dev,y_dev,g_dev))
    assert all(not set(g_dev.iloc[a])&set(g_dev.iloc[b]) for a,b in splits)
    r,_=cv_summary(winner,clone(winner_model),splits); repeat_rows.append({'seed':seed,**r})
repeat_table=pd.DataFrame(repeat_rows); display(repeat_table[['seed','cv_f1_mean','cv_recall_mean','cv_f1_std']]); repeat_table.to_csv(REPORTS/'winner_repeat_cv.csv',index=False)

### Step 059 — Review retrained winner errors

**What and why:** Repeat OOF diagnostic plots and inspect subgroup error counts.

**Checkpoint:** Compare with the baseline diagnostics; no guaranteed improvement.

In [ ]:
tuned_oof,tuned_errors=analyze_oof(winner,winner_model,'retrained_winner')

### Step 060 — Estimate cross-validated permutation importance

**What and why:** Shuffle one raw feature at a time on validation folds; measure the F1 decrease.

**Checkpoint:** Feature importance plot; negative or correlated contributions need cautious interpretation.

In [ ]:
imp=[]
for tr,va in cv:
    fold_model=clone(winner_model).fit(X_dev.iloc[tr],y_dev.iloc[tr])
    result=permutation_importance(fold_model,X_dev.iloc[va],y_dev.iloc[va],scoring='f1',n_repeats=5,random_state=SEED,n_jobs=N_JOBS)
    imp.append(result.importances_mean)
importance=pd.DataFrame({'feature':FEATURES,'mean_cv_permutation':np.mean(imp,axis=0),'fold_sd':np.std(imp,axis=0,ddof=1)}).sort_values('mean_cv_permutation')
importance.to_csv(REPORTS/'feature_importance.csv',index=False)
plt.figure(figsize=(8,6)); plt.barh(importance.feature,importance.mean_cv_permutation,xerr=importance.fold_sd); plt.xlabel('Decrease in CV F1 when shuffled'); save_plot('permutation_importance')

### Step 061 — Freeze the candidate before final evaluation

**What and why:** Record the winner, model URI, dataset hash and predeclared thresholds.

**Checkpoint:** FROZEN printed; do not change candidate based on the holdout.

In [ ]:
freeze={'winner':winner,**winner_ref,'selection':'development CV F1, recall, fold SD','thresholds':{'f1':MIN_F1,'recall':MIN_RECALL,'balanced_accuracy':MIN_BALANCED_ACCURACY},'data_hash':manifest['csv_sha256']}
freeze_path=RUN/'frozen_candidate.json'
if freeze_path.exists(): assert json.loads(freeze_path.read_text())==freeze, 'Candidate already frozen for this run; do not reselect after holdout.'
else: write_json(freeze_path,freeze)
print('FROZEN:',freeze)

## 14 · Final evaluation — one frozen candidate, one holdout
Only now score the untouched holdout. Generate metrics, errors, a group-bootstrap interval and model card. Group bootstrap resamples feature profiles rather than individual duplicate rows. Small sample intervals are illustrative. The gate blocks registration and deployment if metrics fail. Do not tune on this test set or lower thresholds after seeing its results. A later improvement needs new independent data for final evaluation.

### Step 062 — Evaluate the frozen candidate on the holdout

**What and why:** Calculate final predictions and probabilities once for the chosen pipeline.

**Checkpoint:** This is the first final evaluation, not tuning data.

In [ ]:
assert json.loads(freeze_path.read_text())==freeze
pred=winner_model.predict(X_test); prob=winner_model.predict_proba(X_test)[:,1]
holdout=metrics(y_test,pred,prob)
# Bootstrap by feature-profile group; stratification is not forced, omit single-class samples.

### Step 063 — Estimate group-bootstrap uncertainty

**What and why:** Resample feature groups to estimate approximate 95% intervals for F1 and recall.

**Checkpoint:** Interpret intervals as limited by small, nonrepresentative data.

In [ ]:
rng=np.random.default_rng(SEED); labels=g_test.unique(); group_idx={g:np.flatnonzero(g_test.to_numpy()==g) for g in labels}
boot=[]
for _ in range(500):
    idx=np.concatenate([group_idx[g] for g in rng.choice(labels,len(labels),replace=True)])
    yt=y_test.to_numpy()[idx]
    if len(np.unique(yt))==2: boot.append([f1_score(yt,pred[idx],zero_division=0),recall_score(yt,pred[idx],zero_division=0)])
interval=np.quantile(np.asarray(boot),[.025,.975],axis=0)

### Step 064 — Apply the educational acceptance gate

**What and why:** Compare holdout metrics with thresholds set at the start, and log the final evaluation.

**Checkpoint:** Gate True is needed to continue. False means stop; do not retune on the same holdout.

In [ ]:
gate=bool(holdout['f1']>=MIN_F1 and holdout['recall']>=MIN_RECALL and holdout['balanced_accuracy']>=MIN_BALANCED_ACCURACY)
final_report={'candidate':freeze,'metrics':holdout,'holdout_rows':len(y_test),'confusion_matrix':confusion_matrix(y_test,pred,labels=[0,1]).tolist(),
              'bootstrap_95_percent':{'f1':interval[:,0].tolist(),'recall':interval[:,1].tolist(),'valid_resamples':len(boot)},'gate_passed':gate}
write_json(REPORTS/'holdout_report.json',final_report)
with mlflow.start_run(experiment_id=FINAL_EXP,run_name=f'holdout-{winner}'):
    mlflow.set_tags({'source_run_id':winner_ref['run_id'],'source_model_uri':winner_ref['model_uri'],'gate_passed':str(gate)})
    mlflow.log_metrics({f'holdout_{k}':float(v) for k,v in holdout.items()})
    mlflow.log_dict(final_report,'evaluation/holdout_report.json')
checkpoint_mlflow(); display(pd.Series(holdout)); print('Gate:',gate); print('95% bootstrap intervals:',interval)
plt.figure(figsize=(4,3)); sns.heatmap(confusion_matrix(y_test,pred,labels=[0,1]),annot=True,fmt='d'); plt.xlabel('Predicted'); plt.ylabel('Actual'); save_plot('holdout_confusion')
assert gate, 'Educational evaluation gate failed. Stop deployment; do not tune on this holdout.'

## 15 · MLflow Registry and alias
Register the frozen passing model as a version, assign `candidate`, and verify that alias loading reproduces the selected model. Registry alias is mutable; the Bento export is pinned to the resolved model version and Bento tag. This notebook uses one isolated registry per run, so it does not accidentally repoint a shared production model.

### Step 065 — Register the approved model version

**What and why:** A registry entry names a versioned model; tag it with purpose and data lineage.

**Checkpoint:** A registered version is returned.

In [ ]:
assert gate
REGISTRY_NAME='Diabetes-Risk-Educational'
version=mlflow.register_model(winner_ref['model_uri'],REGISTRY_NAME,await_registration_for=120)
client.set_model_version_tag(REGISTRY_NAME,version.version,'dataset_sha256',manifest['csv_sha256'])
client.set_model_version_tag(REGISTRY_NAME,version.version,'purpose','educational-only')

### Step 066 — Assign and resolve the candidate alias

**What and why:** An alias is a movable pointer. Verify it resolves to the exact intended version.

**Checkpoint:** The resolved version matches.

In [ ]:
client.set_registered_model_alias(REGISTRY_NAME,'candidate',version.version)
alias_uri=f'models:/{REGISTRY_NAME}@candidate'
resolved=client.get_model_version_by_alias(REGISTRY_NAME,'candidate')
assert str(resolved.version)==str(version.version)

### Step 067 — Load by alias and compare predictions

**What and why:** Verify the registry model has identical predictions and scores to the frozen candidate.

**Checkpoint:** Registry metadata and checkpoint are saved.

In [ ]:
registry_model=mlflow.sklearn.load_model(alias_uri)
assert np.array_equal(registry_model.predict(X_test),pred)
assert np.allclose(registry_model.predict_proba(X_test),prob)
registry_meta={'registered_name':REGISTRY_NAME,'version':str(version.version),'alias':'candidate','model_uri':alias_uri,'source':winner_ref}
write_json(REPORTS/'registry.json',registry_meta); checkpoint_mlflow(); print(registry_meta)

### Optional MLflow UI (local only)
The UI is optional. Colab does not expose localhost directly in the ordinary browser; the notebook can display an iframe through Colab's supported proxy, but do not expose MLflow publicly. The stored metrics tables above work without a UI. Stop the UI before copying/restoring the database for other use.

### Step 068 — Optional: open MLflow UI

**What and why:** Set RUN_MLFLOW_UI to True only if you want to inspect local runs via Colab. Keep the service bound to localhost.

**Checkpoint:** If disabled, no output is expected; MLflow logging still works.

In [ ]:
RUN_MLFLOW_UI=False
if RUN_MLFLOW_UI:
    ui_log=open(LOCAL/'mlflow_ui.log','w')
    ui_proc=subprocess.Popen(['mlflow','ui','--backend-store-uri',TRACKING_URI,'--host','127.0.0.1','--port','5000'],stdout=ui_log,stderr=subprocess.STDOUT)
    from google.colab import output
    output.serve_kernel_port_as_window(5000)

## 16 · BentoML init and registry model import
BentoML was installed in section 01; verify or install if missing. Download the exact registry version to a local MLflow model directory, then import into BentoML. The exported Bento model contains its own copied MLflow model artifacts and does not need access to the tracking database at serving time. Test hard prediction consistency before creating the API.

### Step 069 — Check and import BentoML

**What and why:** Install only if absent, confirm version, and enforce the educational gate.

**Checkpoint:** Correct BentoML version.

In [ ]:
import importlib.util
if importlib.util.find_spec('bentoml') is None:
    subprocess.run([sys.executable,'-m','pip','install','bentoml==1.4.27'],check=True)
import bentoml, bentoml.mlflow
assert metadata.version('bentoml')=='1.4.27'
assert gate

### Step 070 — Import the exact registry version into BentoML

**What and why:** Download the registered model to local disk and import it as a Bento model with its own tag. Use version, not a mutable alias.

**Checkpoint:** BENTO_TAG identifies the exported artifact.

In [ ]:
pinned_uri=f'models:/{REGISTRY_NAME}/{version.version}'
mlflow_dir=mlflow.artifacts.download_artifacts(artifact_uri=pinned_uri,dst_path=str(LOCAL))
bento_model=bentoml.mlflow.import_model('diabetes_risk_educational',model_uri=str(mlflow_dir),
                         labels={'model_family':winner,'registry_version':str(version.version)})
BENTO_TAG=str(bento_model.tag)

### Step 071 — Verify the imported model

**What and why:** Load Bento model and compare its output with the original registry model.

**Checkpoint:** Prediction equality assertions pass and model store is displayed.

In [ ]:
serving_model=bentoml.mlflow.load_model(BENTO_TAG)
assert np.array_equal(np.asarray(serving_model.predict(X_test)),pred)
assert np.array_equal(np.asarray(serving_model.predict(fixture_frame)),registry_model.predict(fixture_frame))
print('Bento tag:',BENTO_TAG); print('Fixture prediction:',serving_model.predict(fixture_frame))
display(pd.DataFrame({'tag':[str(m.tag) for m in bentoml.models.list()]}))

## 17 · Generate a typed BentoML service in Colab
The API accepts the original 16 fields with strict categorical choices and bounded age. It applies preprocessing and the selected classifier together. The model reference is an immutable Bento tag, not `latest`. Service responses include the selected family and registry version. Inference remains local and is not clinical advice.

### Step 072 — Write the entire BentoML service

**What and why:** Read the multiline Python template: Pydantic validates raw inputs, BentoModel resolves the artifact, and predict wraps a single record in a DataFrame.

**Checkpoint:** service.py is generated, syntax checked and printed.

In [ ]:
service_template = """import bentoml
import bentoml.mlflow
import pandas as pd
from typing import Literal
from pydantic import BaseModel, Field, ConfigDict

YesNo = Literal["Yes", "No"]

class PatientInput(BaseModel):
    model_config = ConfigDict(extra="forbid")
    age: float = Field(ge=1, le=120, allow_inf_nan=False)
    gender: Literal["Male", "Female"]
    polyuria: YesNo
    polydipsia: YesNo
    sudden_weight_loss: YesNo
    weakness: YesNo
    polyphagia: YesNo
    genital_thrush: YesNo
    visual_blurring: YesNo
    itching: YesNo
    irritability: YesNo
    delayed_healing: YesNo
    partial_paresis: YesNo
    muscle_stiffness: YesNo
    alopecia: YesNo
    obesity: YesNo

@bentoml.service
class DiabetesClassifier:
    model_ref = bentoml.models.BentoModel(__TAG__)

    def __init__(self):
        self.model = bentoml.mlflow.load_model(self.model_ref)

    @bentoml.api
    def predict(self, input_data: PatientInput) -> dict:
        frame = pd.DataFrame([input_data.model_dump()])
        frame["age"] = frame["age"].astype("float64")
        result = int(self.model.predict(frame)[0])
        return {"prediction": result, "label": "Positive" if result == 1 else "Negative",
                "model": __MODEL_LABEL__, "registry_version": __VERSION__}
"""
assert gate
service_code=(service_template.replace('__TAG__',repr(BENTO_TAG))
              .replace('__MODEL_LABEL__',repr(winner+'-Educational'))
              .replace('__VERSION__',repr(str(version.version))))
compile(service_code,'service.py','exec')
(DEPLOY/'service.py').write_text(service_code,encoding='utf-8')
print(service_code)

## 18 · Create deterministic API fixtures and automated tests
Three synthetic profiles (not actual patients) are evaluated directly by the frozen model. The expected responses are frozen into a golden fixture; the API must match them before publication. Tests include invalid age, unknown symptom, missing field, extra field, wrong method, repeated requests and readiness. A validation status must be 400 or 422 and identify the relevant field. A 500 is never accepted as input rejection.

### Step 073 — Build golden API fixtures

**What and why:** Generate three synthetic input profiles and expected responses from the registered model. These test serving parity, not diagnostic accuracy.

**Checkpoint:** Three golden cases and test_request.json saved.

In [ ]:
profiles=[fixture_record.copy(),{'age':28.0,'gender':'Male',**{s:'No' for s in SYMPTOMS}},
          {'age':65.0,'gender':'Female',**{s:'Yes' for s in SYMPTOMS}}]
fixture_cases=[]
for i,record in enumerate(profiles,1):
    frame=pd.DataFrame([record],columns=FEATURES); frame.age=frame.age.astype('float64')
    p=int(registry_model.predict(frame)[0])
    fixture_cases.append({'name':f'synthetic-{i}','request':{'input_data':record},
                          'expected':{'prediction':p,'label':'Positive' if p==1 else 'Negative',
                                      'model':winner+'-Educational','registry_version':str(version.version)}})
write_json(DEPLOY/'test_request.json',fixture_cases[0]['request'])
write_json(DEPLOY/'api_fixtures.json',fixture_cases)
display(pd.DataFrame([{'case':x['name'],**x['expected']} for x in fixture_cases]))

### Step 074 — Inspect an actual request and expected response

**What and why:** JSON is the wire contract. The nested input_data key matches the BentoML predict method argument.

**Checkpoint:** Request includes all features and response includes the model version.

In [ ]:
print('POST /predict request:')
print(json.dumps(fixture_cases[0]['request'], indent=2))
print('Expected response:')
print(json.dumps(fixture_cases[0]['expected'], indent=2))

### Step 075 — Write automated API regression tests

**What and why:** Read the full test script. It waits for readiness, checks golden outputs, rejects invalid fields, checks HTTP method, and repeats predictions.

**Checkpoint:** test_api.py compiles; it runs after server startup.

In [ ]:
(DEPLOY/"test_api.py").write_text('''"""Standard-library API regression tests; run with API_BASE_URL=... python3 test_api.py."""
import copy
import json
import os
import sys
import time
import urllib.request
import urllib.error
from pathlib import Path

HERE = Path(__file__).resolve().parent
BASE_URL = os.getenv("API_BASE_URL", "http://127.0.0.1:3001").rstrip("/")


def request(method, path, payload=None):
    data = json.dumps(payload).encode() if payload is not None else None
    req = urllib.request.Request(BASE_URL + path, data=data, method=method,
                                 headers={"Content-Type": "application/json"})
    start = time.perf_counter()
    try:
        with urllib.request.urlopen(req, timeout=20) as response:
            return response.status, response.read().decode(), time.perf_counter()-start
    except urllib.error.HTTPError as error:
        return error.code, error.read().decode(), time.perf_counter()-start


def wait_ready(seconds=180):
    end = time.monotonic()+seconds
    while time.monotonic() < end:
        try:
            if request("GET", "/readyz")[0] == 200:
                return True
        except (OSError, urllib.error.URLError):
            pass
        time.sleep(2)
    return False


def correct_result(status, body, expected):
    if status != 200:
        return False
    try:
        result = json.loads(body)
        return type(result.get("prediction")) is int and result == expected
    except (ValueError, AttributeError):
        return False


def rejects_field(status, body, field):
    if status not in (400, 422):
        return False
    try:
        details = json.loads(body).get("detail", [])
        return isinstance(details, list) and any(
            isinstance(item, dict) and field in item.get("loc", []) for item in details)
    except (ValueError, TypeError, AttributeError):
        return False


def main():
    fixtures = json.loads((HERE / "api_fixtures.json").read_text())
    results = []
    def check(name, passed, detail=""):
        results.append(bool(passed))
        print(f"{'PASS' if passed else 'FAIL'}: {name}" + (f" | {detail}" if detail else ""))
    print("Testing:", BASE_URL)
    check("API readiness", wait_ready())
    if not results[0]:
        print("API did not become ready")
        return 1
    for case in fixtures:
        status, body, elapsed = request("POST", "/predict", case["request"])
        check(case["name"] + " golden prediction", correct_result(status, body, case["expected"]),
              f"HTTP {status}, {elapsed*1000:.1f} ms" + (f" body={body[:300]}" if status != 200 else ""))
    sample = fixtures[0]["request"]
    changes = [("negative age", "age", -5), ("age above limit", "age", 121),
               ("unknown symptom", "polyuria", "Maybe"), ("unknown gender", "gender", "Other"),
               ("extra field", "not_a_feature", "unexpected")]
    for name, field, value in changes:
        payload = copy.deepcopy(sample); payload["input_data"][field] = value
        status, body, _ = request("POST", "/predict", payload)
        check(name, rejects_field(status, body, field), f"HTTP {status}: {body[:250]}")
    payload = copy.deepcopy(sample); del payload["input_data"]["age"]
    status, body, _ = request("POST", "/predict", payload)
    check("missing age", rejects_field(status, body, "age"), f"HTTP {status}")
    status, _, _ = request("GET", "/predict")
    check("GET prediction rejected", status == 405, f"HTTP {status}")
    repeated = [request("POST", "/predict", sample) for _ in range(5)]
    check("five repeat predictions consistent", all(correct_result(s,b,fixtures[0]["expected"]) for s,b,_ in repeated))
    print(f"RESULT: {sum(results)}/{len(results)} tests passed")
    return 0 if all(results) else 1

if __name__ == "__main__":
    sys.exit(main())
''',encoding="utf-8")
compile((DEPLOY/"test_api.py").read_text(),"test_api.py","exec")

## 19 · Start the BentoML server in Colab, send requests and analyze responses
The server runs as a child process bound to loopback, no external exposure. Wait for readiness rather than assuming startup is instantaneous. Inspect JSON and HTTP status, run regression tests, and compare a few sequential request durations. This is an API experiment, not a monitoring implementation. Always stop the child process when finished.

### Step 076 — Start the BentoML server in a child process

**What and why:** Use localhost and port 3000, save stdout/stderr to a log, and start only this notebook’s service.

**Checkpoint:** Process starts; readiness is checked next.

In [ ]:
import signal
API_PORT=3000
API_URL=f'http://127.0.0.1:{API_PORT}'
assert shutil.which('bentoml'), 'BentoML CLI not on PATH'
if 'server_proc' in globals() and server_proc.poll() is None:
    os.killpg(server_proc.pid,signal.SIGTERM); server_proc.wait(timeout=30)
server_log=open(LOCAL/'bento_server.log','w')
server_proc=subprocess.Popen(['bentoml','serve','service:DiabetesClassifier','--host','127.0.0.1','--port',str(API_PORT)],
                              cwd=DEPLOY,stdout=server_log,stderr=subprocess.STDOUT,start_new_session=True)

### Step 077 — Wait for API readiness

**What and why:** Poll /readyz until HTTP 200, or show server logs if the child exits or the timeout expires.

**Checkpoint:** BentoML ready printed. If not, inspect logs before proceeding.

In [ ]:
ready=False
for _ in range(90):
    if server_proc.poll() is not None: break
    try:
        if requests.get(API_URL+'/readyz',timeout=2).status_code==200: ready=True; break
    except requests.RequestException: pass
    time.sleep(2)
if not ready: print((LOCAL/'bento_server.log').read_text()[-6000:])
assert ready, 'BentoML failed readiness; inspect server log.'
print('BentoML ready:',API_URL)

### Step 078 — Send one request manually from Colab

**What and why:** requests.get checks readiness and requests.post sends the JSON body. Inspect response headers and output before running the batch experiments.

**Checkpoint:** HTTP 200 and golden result match.

In [ ]:
r = requests.get(API_URL + '/readyz', timeout=10)
print('Readiness:', r.status_code, repr(r.text))
r = requests.post(API_URL + '/predict', json=fixture_cases[0]['request'], timeout=20)
print('Prediction:', r.status_code, r.headers.get('Content-Type'), r.text)
assert r.status_code == 200 and r.json() == fixture_cases[0]['expected']

### Step 079 — Experiment with valid API requests

**What and why:** Send three synthetic requests three times, record response status, model version, prediction and timing.

**Checkpoint:** Nine response rows and latency summary; latency is descriptive, not a load test.

In [ ]:
import copy
observations=[]
for case in fixture_cases:
    for attempt in range(1,4):
        t=time.perf_counter(); r=requests.post(API_URL+'/predict',json=case['request'],timeout=20); elapsed=(time.perf_counter()-t)*1000
        r.raise_for_status(); body=r.json(); assert body==case['expected']
        observations.append({'case':case['name'],'attempt':attempt,'status':r.status_code,'latency_ms':elapsed,**body})
api_analysis=pd.DataFrame(observations); display(api_analysis)
display(api_analysis.groupby('case').latency_ms.agg(['min','median','max']))
api_analysis.to_csv(REPORTS/'api_experiment.csv',index=False)

### Step 080 — Experiment with invalid input

**What and why:** Send age -5 and inspect Pydantic error body. BentoML may return 400 or 422.

**Checkpoint:** Client error, not HTTP 200 or 500.

In [ ]:
bad=copy.deepcopy(fixture_cases[0]['request']); bad['input_data']['age']=-5
r=requests.post(API_URL+'/predict',json=bad,timeout=20); print('Invalid request:',r.status_code,r.text)
assert r.status_code in (400,422)

### Step 081 — Run the entire regression suite

**What and why:** Invoke the generated test script against the actual Colab server and capture output.

**Checkpoint:** RESULT: 12/12 tests passed; nonzero exit stops packaging.

In [ ]:
run_tests=subprocess.run([sys.executable,str(DEPLOY/'test_api.py')],cwd=DEPLOY,
                         env={**os.environ,'API_BASE_URL':API_URL},capture_output=True,text=True)
print(run_tests.stdout,run_tests.stderr)
( REPORTS/'colab_api_test.txt').write_text(run_tests.stdout+run_tests.stderr)
assert run_tests.returncode==0
API_TEST_PASS=True

### Step 082 — Stop the notebook server safely

**What and why:** Terminate only the child process group, close logs, and back up experiment metadata.

**Checkpoint:** Local serving experiment finished.

In [ ]:
# Stop only the process group launched by this notebook.
if server_proc.poll() is None:
    os.killpg(server_proc.pid,signal.SIGTERM)
    try: server_proc.wait(timeout=20)
    except subprocess.TimeoutExpired:
        os.killpg(server_proc.pid,signal.SIGKILL); server_proc.wait()
server_log.close()
shutil.copy2(LOCAL/'bento_server.log',REPORTS/'bento_server.log')
if 'ui_proc' in globals() and ui_proc.poll() is None:
    ui_proc.terminate(); ui_proc.wait(timeout=30); ui_log.close()
checkpoint_mlflow()
print('Local serving experiment finished.')

## 20 · Preserve the model and generate serving requirements
Export the exact verified Bento model, save the checksum and provenance. Build a minimal top-level dependency manifest from the training environment with the explicitly recorded model requirements. Transitive dependencies are not fully hash-locked: the Docker `pip check`, golden tests and image digest are the release checks. Do not substitute an untrusted model archive. The published image may contain learned information from this public dataset; do not use these steps with confidential datasets without review.

### Step 083 — Export Bento model for Docker

**What and why:** Export the exact tested Bento tag, not a fresh model. Refuse silent overwrite.

**Checkpoint:** diabetes_model.bentomodel exists.

In [ ]:
assert gate and API_TEST_PASS
archive=DEPLOY/'diabetes_model.bentomodel'
assert not archive.exists(), 'Archive already exists; do not silently replace an exported model.'
exported=bentoml.models.export_model(BENTO_TAG,str(archive))
assert archive.is_file(), exported

### Step 084 — Write requirements and model manifest

**What and why:** Pin actual serving package versions and save model and data hashes, evaluation summary and source version.

**Checkpoint:** Requirements and JSON manifests saved.

In [ ]:
(DEPLOY/'requirements.txt').write_text('\n'.join(SERVING_PINS)+'\n')
provenance={'dataset_sha256':manifest['csv_sha256'],'model_archive_sha256':sha256_file(archive),
            'registry':registry_meta,'bento_tag':BENTO_TAG,'algorithm':winner,'educational_gate_passed':gate,
            'holdout_metrics':holdout,'training_python':sys.version,'dependencies':VERSIONS,'run_id':RUN_ID}
write_json(DEPLOY/'model_manifest.json',provenance)
write_json(DEPLOY/'evaluation_summary.json',final_report)
print(exported, archive.stat().st_size, provenance['model_archive_sha256'])

### Step 085 — Create model artifact verifier

**What and why:** Validate the artifact checksum, evaluation gate and fixture count before builds. This detects corruption, not malicious replacement of both model and manifest.

**Checkpoint:** Model manifest and fixture integrity verified.

In [ ]:
(DEPLOY/"verify_artifact.py").write_text("""import hashlib, json
from pathlib import Path
p = Path(__file__).resolve().parent
m = json.loads((p / 'model_manifest.json').read_text())
assert m['educational_gate_passed'] is True
assert hashlib.sha256((p / 'diabetes_model.bentomodel').read_bytes()).hexdigest() == m['model_archive_sha256'], 'Model artifact checksum mismatch'
assert len(json.loads((p / 'api_fixtures.json').read_text())) == 3
print('Model manifest and fixture integrity verified:', m['bento_tag'])
""",encoding="utf-8")
subprocess.run([sys.executable,str(DEPLOY/"verify_artifact.py")],check=True)

## 21 · Docker — layers, image, container and Compose
A Dockerfile is a recipe; each filesystem-changing instruction creates/cache-reuses a layer. An image is the built artifact; a container is a runnable instance. `COPY requirements.txt` precedes `COPY service.py` so code changes usually reuse the dependency layer. We import the Bento archive during build and run as a non-root user. The system uses same Python major/minor as training. Compose publishes only loopback port 3001 and checks `/readyz`. A health check is a deployment gate, not a model quality metric. CI uses the same files you use locally.

### Step 086 — Write Dockerfile layer by layer

**What and why:** FROM sets Python base; WORKDIR sets location; COPY requirements and RUN pip create a cacheable dependency layer; non-root USER imports the model; CMD starts BentoML.

**Checkpoint:** Dockerfile is generated; no image built yet.

In [ ]:
docker_template="""FROM python:__PYTHON__-slim
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1 PIP_NO_CACHE_DIR=1 \\
    BENTOML_HOME=/app/bentoml BENTOML_DO_NOT_TRACK=True
WORKDIR /app
COPY requirements.txt ./
RUN python -m pip install --no-cache-dir -r requirements.txt && python -m pip check
RUN useradd --create-home --uid 10001 appuser && chown appuser:appuser /app
COPY --chown=appuser:appuser diabetes_model.bentomodel ./
COPY --chown=appuser:appuser model_manifest.json api_fixtures.json verify_artifact.py ./
USER appuser
RUN python verify_artifact.py && bentoml models import ./diabetes_model.bentomodel && rm ./diabetes_model.bentomodel
COPY --chown=appuser:appuser service.py ./
EXPOSE 3000
CMD ["bentoml", "serve", "service:DiabetesClassifier", "--host", "0.0.0.0", "--port", "3000"]
"""
(DEPLOY/'Dockerfile').write_text(docker_template.replace('__PYTHON__',f'{sys.version_info.major}.{sys.version_info.minor}'))

### Step 087 — Write Compose configuration

**What and why:** DIABETES_IMAGE selects registry or local image; host 3001 maps to container 3000, readiness probes are internal and resources are limited.

**Checkpoint:** compose.yaml saved; localhost-only exposure.

In [ ]:
(DEPLOY/'compose.yaml').write_text("""name: diabetes-mlops
services:
  diabetes-api:
    image: ${DIABETES_IMAGE:-diabetes-bentoml:v1}
    pull_policy: never
    ports:
      - "127.0.0.1:${DIABETES_PORT:-3001}:3000"
    restart: unless-stopped
    mem_limit: 1g
    cpus: 1.0
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://127.0.0.1:3000/readyz', timeout=5)"]
      interval: 10s
      timeout: 8s
      retries: 6
      start_period: 60s
""")

### Step 088 — Limit build context and Git contents

**What and why:** Allowlist only files required in the image; exclude temporary data, secrets and local experiment outputs from Git.

**Checkpoint:** Ignore files saved and Docker/Compose text displayed.

In [ ]:
(DEPLOY/'.dockerignore').write_text("""*
!Dockerfile
!requirements.txt
!service.py
!diabetes_model.bentomodel
!model_manifest.json
!api_fixtures.json
!verify_artifact.py
""")
(DEPLOY/'.gitignore').write_text("""__pycache__/
*.py[cod]
.env
.env.*
!.env.example
.venv/
*.pem
*.key
*.sqlite
*.db
*.zip
raw/
processed/
extracted/
mlruns/
mlflow_artifacts/
reports/
.ipynb_checkpoints/
""")
print((DEPLOY/'Dockerfile').read_text()); print((DEPLOY/'compose.yaml').read_text())

## 22 · Optional actual Docker build/run in Colab
Most Colab runtimes do not have a usable Docker daemon. Detect capability instead of trying to install or start a privileged daemon. If Docker is available, explicitly set `RUN_DOCKER_HERE=True`, then build the image, inspect layers, create a container, check health and execute regression tests. Otherwise the exact same smoke script runs in GitHub Actions on an Ubuntu runner. Manual equivalent commands are shown below for learning, not required on your new VM after CI.

### Step 089 — Create smoke test automation

**What and why:** Shell script validates the artifact, starts an isolated Compose project, runs tests and cleans up its test container even on failure.

**Checkpoint:** scripts/smoke.sh saved.

In [ ]:
(DEPLOY/'scripts').mkdir(exist_ok=True)
(DEPLOY/'scripts/smoke.sh').write_text("""#!/usr/bin/env bash
set -Eeuo pipefail
cd "$(dirname "$0")/.."
export DIABETES_IMAGE="${DIABETES_IMAGE:-diabetes-bentoml:v1}"
export DIABETES_PORT="${DIABETES_PORT:-3001}"
export COMPOSE_PROJECT_NAME="${COMPOSE_PROJECT_NAME:-diabetes-ci}"
cleanup() {
  status=$?
  trap - EXIT
  if (( status != 0 )); then docker compose logs --tail 100 || true; fi
  docker compose down --remove-orphans || true
  exit "$status"
}
trap cleanup EXIT
python3 verify_artifact.py
docker compose config -q
docker compose up -d --no-build --wait --wait-timeout 180
docker compose ps
API_BASE_URL="http://127.0.0.1:${DIABETES_PORT}" python3 test_api.py
""")

### Step 090 — Create deployment and rollback automation

**What and why:** Accept an immutable GHCR digest, serialize changes with flock, pull before switching, check the exact running image, and restore the prior image on failure.

**Checkpoint:** deploy.sh saved; no remote deployment executed.

In [ ]:
(DEPLOY/'scripts/deploy.sh').write_text("""#!/usr/bin/env bash
# Trusted lab host only. Accept an immutable GHCR manifest digest, never a mutable tag.
set -Eeuo pipefail
cd "$(dirname "$0")/.."
image="${1:-}"
if [[ ! "$image" =~ ^ghcr\\.io/[a-z0-9._-]+/[a-z0-9._/-]+@sha256:[a-f0-9]{64}$ ]]; then
  echo 'Usage: bash scripts/deploy.sh ghcr.io/owner/repo@sha256:<64 hex>'; exit 2
fi
export COMPOSE_PROJECT_NAME=diabetes-mlops
export DIABETES_PORT="${DIABETES_PORT:-3001}"
export DIABETES_IMAGE="$image"
docker compose config -q
# Serialize lab deployments. Do not run other Compose commands concurrently.
state="${HOME}/.local/state/diabetes-mlops"
mkdir -p "$state"
exec 9>"$state/deploy.lock"
flock -n 9 || { echo 'Another deployment is running'; exit 1; }
old_container=$(docker compose ps -aq diabetes-api)
old_image=''
if [[ -n "$old_container" ]]; then
  old_image=$(docker inspect --format '{{.Config.Image}}' "$old_container")
fi
# Pull before replacing existing service.
docker pull "$image"
expected_id=$(docker image inspect --format '{{.Id}}' "$image")
verify_live() {
  local container actual
  container=$(docker compose ps -q diabetes-api)
  [[ -n "$container" ]] || return 1
  actual=$(docker inspect --format '{{.Image}}' "$container")
  [[ "$actual" == "$expected_id" ]] || return 1
  API_BASE_URL="http://127.0.0.1:${DIABETES_PORT}" python3 test_api.py
}
if docker compose up -d --no-build --wait --wait-timeout 180 && verify_live; then
  printf '%s
' "$image" > "$state/current-image.tmp"
  mv "$state/current-image.tmp" "$state/current-image"
  if [[ -n "$old_image" && "$old_image" != "$image" ]]; then printf '%s
' "$old_image" > "$state/previous-image"; fi
  docker compose ps
  echo "DEPLOYMENT PASSED: $image"
else
  echo 'Deployment failed; collecting logs and restoring prior container image.' >&2
  docker compose logs --tail 100 || true
  if [[ -n "$old_image" ]]; then
    export DIABETES_IMAGE="$old_image"
    if docker compose up -d --no-build --wait --wait-timeout 180; then
      echo 'Previous image restored and ready. Validate old-version golden fixtures separately.' >&2
    else
      echo 'Rollback failed; inspect Compose logs manually.' >&2
    fi
  else
    docker compose down || true
  fi
  exit 1
fi
""")

### Step 091 — Validate shell scripts

**What and why:** Use bash -n to check shell syntax before sending scripts to GitHub.

**Checkpoint:** No syntax error.

In [ ]:
for script in (DEPLOY/'scripts').glob('*.sh'):
    script.chmod(0o755)
    subprocess.run(['bash','-n',str(script)],check=True)

### Step 092 — Check Docker availability and optionally build locally

**What and why:** Colab often lacks a Docker daemon. Leave False for GitHub-hosted CI, or set True on a compatible trusted Docker environment.

**Checkpoint:** Either actual Docker commands pass or a clear deferred message appears.

In [ ]:
RUN_DOCKER_HERE=False
available=bool(shutil.which('docker')) and subprocess.run(['docker','info'],capture_output=True).returncode==0
print('Docker daemon available:',available)
if RUN_DOCKER_HERE and available:
    subprocess.run(['docker','build','-t','diabetes-bentoml:v1','.'],cwd=DEPLOY,check=True)
    subprocess.run(['docker','history','diabetes-bentoml:v1'],check=True)
    subprocess.run(['bash','scripts/smoke.sh'],cwd=DEPLOY,check=True)
else:
    print('Docker creation deferred to GitHub Actions; no build success is claimed in Colab.')

### Docker practice (Ubuntu terminal, optional)
```bash
# From generated repository root, only if you want to practice a manual build:
docker build -t diabetes-bentoml:v1 .
docker history diabetes-bentoml:v1
docker image ls
docker compose up -d --no-build --wait --wait-timeout 180
docker compose ps
API_BASE_URL=http://127.0.0.1:3001 python3 test_api.py
docker compose down
```
Do not run the smoke script against your live production Compose project. Its default project name is `diabetes-ci`, and it removes its own test container and network on exit. Use a different host port if 3001 is already in use.

### Ubuntu practice — understand Docker before automating it
   Only after extracting the exported ZIP in section 25 on your own trusted Docker VM. These are optional manual exercises; use GitHub Actions in section 26 if you want automation immediately.

   **A. Inspect the build inputs**
   ```bash
   pwd
   ls -la
   cat Dockerfile
   cat requirements.txt
   python3 verify_artifact.py
   ```
   Checkpoint: files exist and artifact verifier succeeds.

   **B. Build an image and inspect its layers**
   ```bash
   docker build -t diabetes-bentoml:v1 .
   docker image ls diabetes-bentoml
   docker history diabetes-bentoml:v1
   ```
   `-t` assigns a tag; `.` is the build context. The build creates layers; it does not create a running container. Checkpoint: build exits successfully and history shows layer commands.

   **C. Create a standalone container**
   ```bash
   docker run -d --name diabetes-manual -p 127.0.0.1:3000:3000 diabetes-bentoml:v1
   docker ps
   docker logs --tail 30 diabetes-manual
   API_BASE_URL=http://127.0.0.1:3000 python3 test_api.py
   ```
   `-d` means detached; `--name` precedes the image; `-p` publishes ports. Checkpoint: 12/12 tests. If port/name exists, inspect existing resources; do not remove unrelated containers.

   **D. Remove only the practice container and use Compose instead**
   ```bash
   docker stop diabetes-manual
   docker rm diabetes-manual
   docker compose config -q
   docker compose up -d --no-build --wait --wait-timeout 180
   docker compose ps
   API_BASE_URL=http://127.0.0.1:3001 python3 test_api.py
   docker compose down
   ```
   Checkpoint: healthy and 12/12, then Compose removes its own resources. Docker image remains. After this, CI automates the build and smoke test.

## 23 · Continuous integration and continuous delivery
Two workflows: pull requests build and test with a read-only token; a push to `main` builds once, runs the same smoke tests, and publishes **that tested image** to GHCR only if every check succeeds. It publishes an immutable-to-reference SHA digest and a commit tag. Commit tags are intended for traceability but technically mutable; deploy by digest. The publish job prints the exact reference in the Actions summary. No PAT is needed to publish with GitHub Actions when package permissions allow the workflow's `GITHUB_TOKEN`.

GitHub-hosted Ubuntu is the clean build environment. All CI steps fail fast and leave a clear job status. The official checkout action is pinned to a known commit; review and update dependencies before any real deployment.

### Step 093 — Generate pull request CI workflow

**What and why:** Untrusted PR validation runs on GitHub-hosted runners with read-only contents permission. Verify, build, start, test and clean up.

**Checkpoint:** ci.yml exists; not yet triggered.

In [ ]:
(DEPLOY/'.github/workflows').mkdir(parents=True,exist_ok=True)
(DEPLOY/'.github/workflows/ci.yml').write_text("""name: Pull request CI
on:
  pull_request:
    branches: [main]
permissions:
  contents: read
jobs:
  test:
    runs-on: ubuntu-latest
    timeout-minutes: 25
    steps:
      - uses: actions/checkout@11bd71901bbe5b1630ceea73d27597364c9af683
        with:
          persist-credentials: false
      - name: Verify model and source
        run: |
          python3 verify_artifact.py
          python3 -m py_compile service.py test_api.py verify_artifact.py
          bash -n scripts/smoke.sh scripts/deploy.sh
          docker compose config -q
      - name: Build image
        run: docker build -t diabetes-bentoml:v1 .
      - name: Smoke test container
        run: bash scripts/smoke.sh
""")

### Step 094 — Generate main-branch delivery and optional deployment

**What and why:** Build once, test exact image, publish to GHCR using temporary GITHUB_TOKEN and return an immutable digest. Optional trusted runner deploys only when explicitly enabled.

**Checkpoint:** cd.yml exists; no runner or credentials needed in Colab.

In [ ]:
(DEPLOY/'.github/workflows/cd.yml').write_text("""name: Main CI and GHCR delivery
on:
  push:
    branches: [main]
  workflow_dispatch:
permissions:
  contents: read
concurrency:
  group: diabetes-main-delivery
  cancel-in-progress: false
jobs:
  publish:
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    timeout-minutes: 30
    permissions:
      contents: read
      packages: write
    outputs:
      image: ${{ steps.publish.outputs.image }}
    steps:
      - uses: actions/checkout@11bd71901bbe5b1630ceea73d27597364c9af683
        with:
          persist-credentials: false
      - name: Verify artifact and scripts
        run: |
          python3 verify_artifact.py
          python3 -m py_compile service.py test_api.py verify_artifact.py
          bash -n scripts/smoke.sh scripts/deploy.sh
          docker compose config -q
      - name: Build image once
        run: docker build --label "org.opencontainers.image.source=https://github.com/$GITHUB_REPOSITORY" --label "org.opencontainers.image.revision=$GITHUB_SHA" -t diabetes-bentoml:v1 .
      - name: Test the exact image
        run: bash scripts/smoke.sh
      - name: Authenticate and publish tested image
        id: publish
        env:
          GHCR_TOKEN: ${{ secrets.GITHUB_TOKEN }}
        shell: bash
        run: |
          set -Eeuo pipefail
          repo="ghcr.io/${GITHUB_REPOSITORY,,}"
          tag="$repo:sha-$GITHUB_SHA"
          export DOCKER_CONFIG="$(mktemp -d)"
          trap 'docker logout ghcr.io >/dev/null 2>&1 || true; rm -rf "$DOCKER_CONFIG"' EXIT
          printf '%s' "$GHCR_TOKEN" | docker login ghcr.io -u "$GITHUB_ACTOR" --password-stdin
          docker tag diabetes-bentoml:v1 "$tag"
          docker push "$tag" | tee push.log
          digest=$(sed -nE 's/.*digest: (sha256:[0-9a-f]{64}).*/\\1/p' push.log | tail -1)
          [[ "$digest" =~ ^sha256:[a-f0-9]{64}$ ]]
          ref="$repo@$digest"
          echo "image=$ref" >> "$GITHUB_OUTPUT"
          printf '## Published tested image\\n\\n`%s`\\n\\nCommit: `%s`\\n' "$ref" "$GITHUB_SHA" >> "$GITHUB_STEP_SUMMARY"
  # Enable only on a dedicated, persistent, trusted host. THM VMs are not suitable.
  deploy:
    needs: publish
    if: vars.ENABLE_LAB_AUTO_DEPLOY == 'true' && github.ref == 'refs/heads/main'
    runs-on: [self-hosted, linux, x64, diabetes-lab]
    timeout-minutes: 20
    environment: lab
    permissions:
      contents: read
      packages: read
    steps:
      - uses: actions/checkout@11bd71901bbe5b1630ceea73d27597364c9af683
        with:
          persist-credentials: false
      - name: Deploy digest with readiness, regression tests and rollback
        env:
          GHCR_TOKEN: ${{ secrets.GITHUB_TOKEN }}
          IMAGE_REF: ${{ needs.publish.outputs.image }}
        shell: bash
        run: |
          set -Eeuo pipefail
          export DOCKER_CONFIG="$(mktemp -d)"
          trap 'docker logout ghcr.io >/dev/null 2>&1 || true; rm -rf "$DOCKER_CONFIG"' EXIT
          printf '%s' "$GHCR_TOKEN" | docker login ghcr.io -u "$GITHUB_ACTOR" --password-stdin
          bash scripts/deploy.sh "$IMAGE_REF"
""")

### Step 095 — Validate workflow YAML

**What and why:** Parse YAML preserving on as a key and print job names. GitHub Actions itself must validate the workflow later.

**Checkpoint:** Both workflow files contain triggers and jobs.

In [ ]:
import yaml
for file in (DEPLOY/'.github/workflows').glob('*.yml'):
    parsed=yaml.load(file.read_text(),Loader=yaml.BaseLoader)  # preserve the YAML key 'on'
    assert 'on' in parsed and 'jobs' in parsed
    print(file.name,':',list(parsed['jobs']))
print((DEPLOY/'.github/workflows/cd.yml').read_text())

## 24 · Deployment mode selection: ephemeral THM VM vs persistent host
**Recommended for this learning exercise:** automated CI and GHCR delivery; one command to pull and deploy on a fresh VM. A temporary TryHackMe machine is not a reliable continuous-deployment destination and may not be appropriate for storing credentials or registering a runner. Use only a VM you are authorized to use for this workload, prefer a personal Ubuntu VM, and do not expose the API externally. Public GHCR packages can be pulled anonymously (only make a package public if you are comfortable sharing its contents). For private packages, authenticate on a trusted personal machine using a minimal read-only package token and `docker login --password-stdin`; do not paste tokens into Colab or a temporary shared host.

**Optional full automatic deployment:** on a dedicated persistent private Ubuntu host, configure a restricted self-hosted GitHub Actions runner using repository Settings → Actions → Runners, with labels `linux`, `x64` and `diabetes-lab`; install Docker and Compose, verify runner account Docker access, create protected `lab` environment and require approval when supported. Protect `main`, restrict who can push workflow changes, and never allow untrusted PR jobs onto the runner. Only then set repository Actions variable `ENABLE_LAB_AUTO_DEPLOY=true`. With it unset, deployment is skipped; GitHub will not queue a nonexistent runner. The `publish` job's outputs are passed directly to the deploy job, not copied manually. Docker access is root-equivalent; do not use your everyday workstation as an untrusted runner. A private package must grant repository access.

The rollback script restores the prior image if readiness or golden tests fail. It does not implement zero downtime or database rollback; there is no application database in this lab. Keep the old version's repository checkout and fixtures for full rollback validation. Do not prune the last known good image before validation.

### Step 096 — Generate repository README

**What and why:** Provide instructions for trusted local deployment, workflow triggers, GHCR and rollback, with limitations.

**Checkpoint:** README.md saved.

In [ ]:
(DEPLOY/"README.md").write_text("""# Diabetes MLOps — educational deployment

Public UCI dataset, educational classification only; not a medical device or diagnosis. This repository contains a trained Bento model, synthetic API fixtures, Docker/Compose deployment and GitHub Actions. Training and private experiment files remain on Google Drive.

## Local (optional)
Prerequisites: Linux/amd64 or compatible platform, Docker Engine, Compose v2 with `--wait`, Python 3, bash. From repository root:

```bash
python3 verify_artifact.py
docker build -t diabetes-bentoml:v1 .
bash scripts/smoke.sh
```

## CI/CD
- PR to `main`: `.github/workflows/ci.yml` builds and tests the image.
- Push to `main`: `.github/workflows/cd.yml` builds, tests, then publishes the tested image to GitHub Container Registry. Use the immutable digest printed in the run summary.
- Optional `ENABLE_LAB_AUTO_DEPLOY=true`: dedicated trusted runner label `diabetes-lab` and protected `lab` environment; never enable on a temporary/shared/untrusted host.

## Deploy a published image (no build)
Clone this repository at the commit reported by the successful Actions run. Authenticate only if package is private and the host is trusted. Copy the digest reference from Actions:

```bash
bash scripts/deploy.sh 'ghcr.io/OWNER/REPO@sha256:REPLACE_WITH_REAL_DIGEST'
docker compose ps
API_BASE_URL=http://127.0.0.1:3001 python3 test_api.py
```

The placeholder is intentionally invalid. Replace it with the real lowercase digest reference. Script validates syntax, pulls the image, starts Compose, checks readiness, verifies running image ID and tests golden responses. If update fails, it attempts to restore previous container image. Rollback is not zero-downtime; use old-version fixtures to validate previous behavior. To stop: `docker compose down`. Never expose the educational service publicly.

## Files
`service.py`, `diabetes_model.bentomodel`, `model_manifest.json`, `evaluation_summary.json`, `requirements.txt`, `Dockerfile`, `compose.yaml`, `api_fixtures.json`, `test_api.py`, `verify_artifact.py`, `scripts/`, `.github/workflows/`.

## Data source and documentation
- https://archive.ics.uci.edu/dataset/529/early+stage+diabetes+risk+prediction+dataset
- https://mlflow.org/docs/latest/ml/mlflow-3
- https://docs.bentoml.org/en/latest/examples/mlflow.html
- https://docs.github.com/en/actions/tutorials/publish-packages/publish-docker-images

No monitoring stack is included. Dependency pins cover direct model and serving requirements, not every transitive dependency. Build, smoke tests and digest verification are required before release. Review dependency security and data/model distribution permissions before publishing.
""",encoding="utf-8")

## 25 · Final package, integrity checks and backup
Create the Git-ready deployment ZIP, containing hidden workflow files, without raw observations, logs with row-level errors, tracking DB, tokens or Colab environment secrets. Save the archive + checksum to Drive. Copy the file into an empty Git repository (avoid mixing with unrelated VM files). The notebook itself can be committed separately after clearing any sensitive outputs. The model is trusted because you trained and exported it in this run.

### Step 097 — Validate all release files

**What and why:** Check required files, Python and shell syntax, artifact integrity, original data hashes and SQLite backup.

**Checkpoint:** All checks must pass before packaging.

In [ ]:
assert gate and API_TEST_PASS
required=['service.py','Dockerfile','compose.yaml','requirements.txt','diabetes_model.bentomodel','model_manifest.json',
          'evaluation_summary.json','api_fixtures.json','test_request.json','test_api.py','verify_artifact.py','README.md',
          '.dockerignore','.gitignore','scripts/smoke.sh','scripts/deploy.sh','.github/workflows/ci.yml','.github/workflows/cd.yml']
assert all((DEPLOY/x).is_file() for x in required)
for x in ['service.py','test_api.py','verify_artifact.py']: compile((DEPLOY/x).read_text(),x,'exec')
subprocess.run([sys.executable,str(DEPLOY/'verify_artifact.py')],check=True)
for x in ['scripts/smoke.sh','scripts/deploy.sh']: subprocess.run(['bash','-n',str(DEPLOY/x)],check=True)
assert sha256_file(raw_csv)==manifest['csv_sha256'] and sha256_file(raw_zip)==raw_hash
checkpoint_mlflow()

### Step 098 — Create the complete Git-ready ZIP

**What and why:** Include hidden workflows using an explicit allowlist. Save a SHA-256 sidecar and inspect the ZIP member list.

**Checkpoint:** Drive export path and checksum printed.

In [ ]:
bundle=RUN/'diabetes-mlops-deployment.zip'
assert not bundle.exists(), 'Bundle already exported. Start a new session for a new release.'
with zipfile.ZipFile(bundle,'x',compression=zipfile.ZIP_DEFLATED) as z:
    for rel in required: z.write(DEPLOY/rel,rel)
with zipfile.ZipFile(bundle) as z:
    assert z.testzip() is None and set(z.namelist())==set(required)
    print('Included files:',*z.namelist(),sep='\n')
( RUN/'deployment.sha256').write_text(f'{sha256_file(bundle)}  {bundle.name}\n')
print('Drive export:',bundle,'\nSHA256:',sha256_file(bundle))
print('Ready for GitHub. CI and deployment checks must still run externally.')

## 26 · GitHub upload and CI/CD execution (external steps)
1. Download the ZIP from Drive; create a **new empty private repository** on GitHub (recommended). Do not initialize it with README if you want to push this existing folder straight away.
2. On your trusted computer, extract the ZIP into a clean folder. On Ubuntu use `python3 -m zipfile -e diabetes-mlops-deployment.zip ./diabetes-mlops` or unzip. Hidden files are included.
3. Inspect files and commit. Replace `YOUR_OWNER` and `YOUR_REPO` in the remote URL. Authenticate using your normal Git credential mechanism or GitHub CLI; never include tokens in the remote URL.

```bash
cd diabetes-mlops
ls -la
python3 verify_artifact.py
git init -b main
git add .
git diff --cached --stat
git commit -m "Complete diabetes MLOps training and deployment"
git remote add origin https://github.com/YOUR_OWNER/YOUR_REPO.git
git push -u origin main
```

4. GitHub → Actions → **Main CI and GHCR delivery**. Verify build, smoke tests and publishing all succeeded. A skipped optional deployment is expected. Copy the `ghcr.io/...@sha256:...` reference from the Actions job summary and note the exact commit SHA. If publishing is denied, inspect package access and permissions rather than broadening tokens unnecessarily.
5. For a public package, only change visibility if you intend to publish your model and are permitted to do so. For private packages, use a trusted machine with authorized package read access; avoid credentials on temporary/shared VMs.
6. Create a feature branch, change a nonfunctional README line, push, open a PR into `main` and verify **Pull request CI** runs. Merge only after successful checks. This demonstrates changes can trigger build and test without issuing Docker commands yourself.

### What automation does now
`git push` → GitHub-hosted runner → model checksum verification → Docker build/layers → Compose container creation → readiness → golden regression tests → GHCR publication. On a dedicated opted-in runner, the tested digest is deployed automatically after publication and required environment approval. Otherwise deployment is one explicit command in the next section.

### GitHub browser checklist — do this in order
   **1. Create the repository:** New repository → choose a name such as diabetes-mlops → Private → leave initialization unchecked. Copy the HTTPS URL. Use your own trusted computer to push, not a shared VM with your long-lived credentials.

   **2. Upload the files:** extract the Drive ZIP, including `.github/`, initialize Git and push using the commands above. Review `git diff --cached --stat` first; there should be no raw data or credentials.

   **3. Define the first automatic run:** open Actions. The `Main CI and GHCR delivery` workflow starts after the push to main. Expand the publish job, watch each step: checkout → artifact verification → Docker build → smoke → publish. A failure stops later dependent steps. Open the failed step to see logs. Fix source, commit and push again; don't manually publish an untested image.

   **4. Collect delivery evidence:** open the successful run summary, copy the exact digest and commit SHA. In repository Packages, verify the image exists. `sha-<commit>` is a tag; `@sha256:...` is the immutable deployment identity.

   **5. Practice pull request CI:** on your trusted checkout, run:
   ```bash
   git switch -c lab/readme-ci
   printf '\nCI practice change.\n' >> README.md
   git add README.md
   git commit -m "Practice pull request CI"
   git push -u origin lab/readme-ci
   ```
   In GitHub open a PR to main, inspect `Pull request CI`, merge only when green. Main workflow runs again and publishes a new tested digest. This shows the difference between PR CI and main delivery. The optional deployment job stays skipped until you explicitly configure a dedicated persistent runner.

   **Checkpoint:** record the actual Actions URL, SHA and digest in section 28. Do not mark the pipeline complete if it failed.

## 27 · Fresh VM deployment (no manual Docker build)
Use a machine you control and are authorized to use. Verify `docker version`, `docker compose version`, `git --version`, `python3 --version`, `df -h /`, `free -h`. It must have sufficient space for the old and new images during rollback (several GiB is prudent). Use a fresh checkout at the commit that produced the digest; for private repo use an authorized trusted machine.

```bash
git clone https://github.com/YOUR_OWNER/YOUR_REPO.git diabetes-mlops
cd diabetes-mlops
git checkout YOUR_PUBLISHED_COMMIT_SHA
# Copy real digest from GitHub Actions; do not run with placeholder.
bash scripts/deploy.sh 'ghcr.io/OWNER/REPO@sha256:REPLACE_WITH_REAL_DIGEST'
docker compose ps
curl -i http://127.0.0.1:3001/readyz
API_BASE_URL=http://127.0.0.1:3001 python3 test_api.py
```
Expected: healthy container, HTTP 200, all 12 contract tests pass. No `docker build` needed: your image came from the registry. The script internally runs Docker pull, Compose and tests. Package and repository can be public independently, so check both access settings.

### Recovery exercise (only on a disposable lab service)
- Keep the previous successful digest and its Git revision. Avoid pruning the previous image.
- Deploy a second successfully published digest and verify its matching fixtures. Deployment script saves current and previous image references in `~/.local/state/diabetes-mlops/`.
- To manually roll back, check out the previous commit (old contract fixtures), and pass its exact digest to `scripts/deploy.sh`. Confirm readiness and old regression tests.
- If a deployment has failed, inspect `docker compose logs --tail 100`; script attempts to restore old image automatically. Invalid digest or authentication failures happen before replacement.
- For an intentional restart exercise: `docker compose restart` then verify readiness and tests. Be aware this causes short downtime.

Do not delete unrelated images or containers on shared machines. A new VM can always pull the published image again.

### Deployment troubleshooting and verification
   On the target VM, `docker info` must succeed. If it doesn't, fix Docker access on a machine you control. Check `df -h /` before pulling. Use a public image only if you intend to share its contents and have permission; never paste credentials into shared or ephemeral machines. On trusted hosts, authenticate privately when needed.

   `scripts/deploy.sh` is the single deployment command: it pulls the exact image, creates/replaces the Compose container, waits for readiness, runs API tests and tries to restore the previous image if tests fail. You do not type `docker build` on the new VM.

   **Verify:** `docker compose ps` should show healthy; `curl` should give 200; `test_api.py` must report 12/12. If port 3001 is occupied, inspect `docker ps` and use another free localhost port via `DIABETES_PORT`; pass the same port to tests. If image pull fails, check repository/package visibility and actual digest. If tests fail, read `docker compose logs --tail 100` and use the previous commit's fixtures for rollback verification.

   **Full automation:** on a dedicated persistent trusted host, follow section 24 to register the labeled runner and enable the deployment variable only after approval and access restrictions. Push a new commit and inspect the optional deploy job. Temporary THM VMs are suitable only if authorized for manual pull/deploy practice, not persistent runners. No monitoring stack is introduced.

## 28 · Completion checklist and release decision
Record actual external outcomes rather than marking them complete in advance. The optional cells are explicit user-entered evidence. Monitoring is the next independent phase and is not included here.

### Step 099 — Record actual CI/CD evidence

**What and why:** Set external flags only after observing results from GitHub and your VM. False is correct before executing external steps.

**Checkpoint:** completion_checklist.json saved; no external success assumed.

In [ ]:
# Fill only AFTER observing each result. Leaving False is correct until you have evidence.
external_evidence={
 'github_ci_passed':False,
 'ghcr_image_published':False,
 'fresh_vm_deployed':False,
 'fresh_vm_api_tests_passed':False,
 'rollback_exercise_passed':False,
 'published_digest':'',
 'github_commit':'',
 'github_actions_run_url':'',
}
progress={'raw_preserved':sha256_file(raw_zip)==raw_hash,'group_split_safe':not bool(set(g_dev)&set(g_test)),
          'five_baselines_logged':len(baseline_refs)==5,'five_metrics_audits':len(baseline_audit)==5,
          'five_retrained_logged':len(tuned_refs)==5,'holdout_gate':gate,'registry_alias_tested':True,
          'bento_import_verified':True,'colab_api_tests_passed':API_TEST_PASS,'deployment_bundle_created':bundle.exists(),
          **external_evidence}
display(pd.Series(progress,name='evidence'))
write_json(RUN/'completion_checklist.json',progress)

## References and boundaries
- [UCI Early Stage Diabetes Risk Prediction](https://archive.ics.uci.edu/dataset/529/early+stage+diabetes+risk+prediction+dataset)
- [MLflow 3 migration and model URIs](https://mlflow.org/docs/latest/ml/mlflow-3)
- [MLflow tracking](https://mlflow.org/docs/latest/tracking/)
- [BentoML MLflow integration](https://docs.bentoml.org/en/latest/examples/mlflow.html)
- [BentoML model import/export](https://docs.bentoml.org/en/latest/build-with-bentoml/model-loading-and-management.html)
- [GitHub Actions publishing Docker images](https://docs.github.com/en/actions/tutorials/publish-packages/publish-docker-images)

No claims of clinical readiness, external test execution or successful deployment are made before observing actual output. Do not reuse the holdout to tune models. Next separate lab: logging/monitoring, drift, Prometheus/Grafana, infrastructure and retraining schedules.